# Soll-Ist-Vergleich HTS-Cargo – Tourenanalyse März 2026

**Ziel:** den geplanten Tourenablauf (Soll, aus dem Planungssystem) mit dem tatsächlich gefahrenen Ablauf (Ist, aus Telematik und GPS) über alle Touren des Monats vergleichen, um Verspätungen, ausgelassene oder zusätzliche Stopps und überlange Standzeiten zu messen.

## Datengrundlage

Das Projekt stützt sich auf eine Plan-Quelle und zwei parallele Ist-Telematik-Kanäle, deren gemeinsamer Anker das normalisierte Fahrzeug-Kennzeichen ist. Die Plan-Quelle ist das CIS-Soll aus dem TMS: die geplanten Touren mit ihren Stopps, Soll-Zeiten und Kundenadressen samt Koordinaten, identifiziert über Tournummer und Stopp-Nummer. Sie ist die Referenz für die geplante Reihenfolge, die Soll-Zeiten und die Orte, und ihre Schwächen sind vereinzelte Geocoding-Fehler und die schon in der Konsolidierung behandelten Planungs-Doppelungen.

Die Ist-Seite kommt aus zwei Telematik-Quellen, die dasselbe Geschehen aus verschiedenen Blickwinkeln aufzeichnen. PRC sind die box-generierten Meldungen des Fahrzeugs und zerfällt in mehrere Meldungstypen: Der Stationsstatus bildet den Zyklus jedes Stopps über eine versionierte StationID ab und ist die einzige Quelle, in der sich Re-Dispositionen zeigen, liefert aber nur für einen Teil der Besuche eine verwertbare Standzeit. Die Position ist die reine GPS-Spur und damit die dichte gefahrene Route während der Fahrt, steht im Stillstand aber naturgemäß still und trägt dort nichts zur Standzeit bei. EMR ist die Telematik der SPEDION-Fahrer-App mit der dichtesten GPS-Spur, die auch im Stand weiterläuft und so die Grundlage für die Standzeit über Geofencing bildet; ihre Statuseingaben hängen dagegen am Fahrer und sind lückiger.

Zwei PRC-Teile bleiben zunächst außen vor. Die FMS-Werte tragen Fahrzeugzustände wie Füllstand, Verbrauch und Service; davon ist nur der Kilometerstand zielnah, und der liegt bereits in der Position vor. Tour- und Sendposstatus klammern die Tour zeitlich und melden Lade- und Entlademengen, werden aber für die vier Kennzahlen vorerst nicht gebraucht.

Konzeptionell unterscheiden sich die drei Quellen: CIS-Soll ist, was geplant war; PRC ist, was die Box server-seitig aufzeichnete, und damit die objektivere Quelle; EMR ist, was die Fahrer-App erfasste, subjektiver in den Statuseingaben, aber mit der dichtesten Spur. Verknüpft werden sie über das Kennzeichen, dazu Soll und PRC über die Tour und PRC und EMR über Fahrzeug und Zeitstempel. Wie die einzelnen gefahrenen Stopps den geplanten Stopps zugeordnet werden, leiten wir in Kapitel 7 her; an dieser Stelle steht nur die Aufgabenteilung, aus der der finale Ist-Datensatz entsteht: Reihenfolge und Re-Disposition aus dem Stationsstatus, Standzeit aus EMR und der verwertbaren Hälfte des Stationsstatus, gefahrene Route aus der Position, alles gegen die CIS-Soll-Planung gespiegelt.

## 1 – Daten laden

Vier Quellen liegen vor, in separaten Notebooks bereinigt und als Pickle gespeichert. Soll ist der geplante Tourenablauf aus dem Planungssystem CIS, EMR sind die app-basierten Ist-Meldungen der Fahrer aus SPEDION, PRC sind die box-generierten Telematik-Meldungen des Fahrzeugs, und fms sind die separat aufbereiteten Fahrzeugzustandswerte aus den PRC-Meldungen. Pickle hält die in der Bereinigung gesetzten Datentypen fest, sodass hier keine erneute Typanpassung nötig ist. Die FMS-Werte halten wir vorerst nur vor für den Fall, dass sich ihre Codes noch klären lassen.

In [41]:
import pandas as pd
from pathlib import Path
import numpy as np

# Anzeigeoptionen, damit DataFrames vollständig sichtbar sind
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", None)

DATA = Path.cwd().parent / "data"
PROCESSED = DATA / "processed"

soll = pd.read_pickle(PROCESSED / "solldaten_cis_clean.pkl")
emr  = pd.read_pickle(PROCESSED / "istdaten_emr_clean.pkl")
prc  = pd.read_pickle(PROCESSED / "istdaten_prc_clean.pkl")
fms  = pd.read_pickle(PROCESSED / "istdaten_prc_fmswerte.pkl")

# Zentrale Hilfsfunktion für alle Distanzberechnungen (Meter zwischen zwei Koordinaten)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Form, Datentypen und Nicht-Null-Werte je Quelle
for name, df in [("soll", soll), ("emr", emr), ("prc", prc), ("fms", fms)]:
    print(f"\n{name}: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
    df.info()
    display(df.head(2))


soll: 1577 Zeilen, 16 Spalten
<class 'pandas.DataFrame'>
RangeIndex: 1577 entries, 0 to 1576
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   TOURNR          1577 non-null   str           
 1   NUMMER          1577 non-null   int64         
 2   TOURSTATION_ID  1577 non-null   int64         
 3   TYP             1577 non-null   int64         
 4   BEZEICHNUNG     1577 non-null   str           
 5   TOURENART       1577 non-null   int64         
 6   STARTDATUM      1577 non-null   datetime64[us]
 7   ANKUNFT         1577 non-null   datetime64[us]
 8   ABFAHRT         1577 non-null   datetime64[us]
 9   NAME1           1577 non-null   str           
 10  ORT             1577 non-null   str           
 11  STRASSE         1577 non-null   str           
 12  GEOX            1577 non-null   float64       
 13  GEOY            1577 non-null   float64       
 14  LKW_KENNZ       1577 non-null   str 

,TOURNR,NUMMER,TOURSTATION_ID,TYP,BEZEICHNUNG,TOURENART,STARTDATUM,ANKUNFT,ABFAHRT,NAME1,ORT,STRASSE,GEOX,GEOY,LKW_KENNZ,FAHRERNAME
0,H/42374,2,553625,4,Greif,0,2026-03-02 06:00:00,2026-03-02 06:00:00,2026-03-02 06:30:00,Greif Packaging Germany GmbH,Hamburg,Brandenburger Straße 12,9.98632,53.52348,PLO-TS-856,"Hesse, Sven-Erik"
1,H/42374,3,553626,8,Greif,0,2026-03-02 06:00:00,2026-03-02 06:49:00,2026-03-02 07:19:00,Shell Deutschland GmbH,Hamburg,Worthdamm 32 Building: Grasbrook Lubrica,9.99383,53.54751,PLO-TS-856,"Hesse, Sven-Erik"



emr: 149119 Zeilen, 10 Spalten
<class 'pandas.DataFrame'>
Index: 149119 entries, 2678 to 187460
Data columns (total 10 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   FAHRERNAME            128335 non-null  str           
 1   Formularnummer        149119 non-null  int64         
 2   Formularbeschreibung  149119 non-null  str           
 3   Meldungszeit          149119 non-null  datetime64[us]
 4   Geschwindigkeit       149119 non-null  int64         
 5   KM-Stand              149119 non-null  float64       
 6   Breitengrad           149119 non-null  float64       
 7   Längengrad            149119 non-null  float64       
 8   Position              149119 non-null  str           
 9   LKW_KENNZ             149119 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(2), str(4)
memory usage: 25.1 MB


,FAHRERNAME,Formularnummer,Formularbeschreibung,Meldungszeit,Geschwindigkeit,KM-Stand,Breitengrad,Längengrad,Position,LKW_KENNZ
2678,NaN,30012,Reboot - Generic Third Party Telematic Device Position M...,2026-03-02 00:22:52,0,314612.18,53.66768,9.99212,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS-8041
2679,NaN,30006,Startup - Generic Third Party Telematic Device Position ...,2026-03-02 00:24:00,0,314612.18,53.66768,9.99212,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS-8041



prc: 29429 Zeilen, 15 Spalten
<class 'pandas.DataFrame'>
Index: 29429 entries, 18 to 33153
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   msg_typ               29429 non-null  str           
 1   quelle_datei          29429 non-null  str           
 2   PositionsID           29429 non-null  str           
 3   Longitude             29429 non-null  float64       
 4   Latitude              29429 non-null  float64       
 5   Geschwindigkeit       29429 non-null  int64         
 6   KMStand               29429 non-null  int64         
 7   Zeitstempel_GPS       29429 non-null  datetime64[us]
 8   Zeitstempel_Fahrzeug  29429 non-null  datetime64[us]
 9   Zeitstempel_Server    29429 non-null  datetime64[us]
 10  LKW_KENNZ             29429 non-null  str           
 11  Status                11562 non-null  Int64         
 12  TourID                1244 non-null   str           
 13  

,msg_typ,quelle_datei,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,LKW_KENNZ,Status,TourID,StationID,SendposID
18,Position,msg_20260301000037_974.imp.20260301000126628.prc,16052606718,9.98615,53.52342,0,502579,2026-02-28 23:59:59,2026-03-01,2026-03-01 00:00:37,PLO-TS-859,<NA>,NaN,NaN,NaN
19,Position,msg_20260301000037_975.imp.20260301000126651.prc,16052611797,9.98547,53.52335,0,339687,2026-02-28 23:59:59,2026-03-01,2026-03-01 00:00:37,PLO-TS-856,<NA>,NaN,NaN,NaN



fms: 33380 Zeilen, 4 Spalten
<class 'pandas.DataFrame'>
RangeIndex: 33380 entries, 0 to 33379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PositionsID  33380 non-null  str    
 1   Typ          33380 non-null  str    
 2   Wert         33380 non-null  float64
 3   LKW_KENNZ    33380 non-null  str    
dtypes: float64(1), str(3)
memory usage: 1.7 MB


,PositionsID,Typ,Wert,LKW_KENNZ
0,16053897222,4,0.0,OD-TS-8046
1,16053897222,1,0.0,OD-TS-8046


Die FMSStatus-Meldungen tragen Fahrzeug-Telemetrie in einem Werte-Block mit nummerierten Typen. Die HTS-Übersetzung weist sie als FuelLevelInPercent (1), VehicleDistanceInKM (2), FuelUsedInLiter (4) und ServiceDistanceInKM (5) aus. Nur der Kilometerstand (Typ 2) berührt ein Projektziel, der liegt jedoch bereits in den Position-Meldungen vor. Bei den drei Fahrzeugen ohne Position-Kilometerstand fehlt auch der FMS-Kilometerstand. Wir legen die FMS-Werte daher beiseite und führen sie nicht weiter mit, sie könnten allerdings für weitere Analysen verwendet werden..

# 2 – PRC-Statuscodes übersetzen

Der Status einer PRC-Meldung ist ein Code, dessen Bedeutung vom Meldungstyp abhängt. Festgelegt sind diese Codes in der C-LOGISTIC-Schnittstellenbeschreibung, mit je einer Codetabelle für Stations-, Sendpos- und Tourstatus. CIS importiert genau diese Codes und zeigt sie in der Oberfläche als Klartext an.

## Bedeutung der Codes: Norm statt XML-Text

Naheliegend war zuerst, die Texte aus dem C-LOGISTIC-Mapping-XML zu ziehen, das wir von Spedion bekommen haben, denn dort steht zu jeder Kombination aus Meldungstyp und Status ein StatusText. Beim Abgleich mit der CIS-Oberfläche bzw. den konkreten Statusmeldungen einer Tour, hat sich aber gezeigt, dass CIS die Codes nicht über diese XML-Texte anzeigt, sondern über die Norm-Bedeutung des Codes. Das Problem dabei ist, dass beide Übersetzungsquellen unterschiedliche Bedeutungen für den gleichen Code tragen. Am deutlichsten z.B. beim Stationscode 11. Das XML nennt ihn als angenommen, CIS hingegen zeigt Anfahrt. Auch bei den Tourcodes weicht es ab, etwa XML Vorlaeufiges Tourende gegen CIS Beendet und XML Daten zum Endgerät übertragen gegen CIS Empfangen vom Fahrzeug.

Belegen lässt sich das also am Besten über die CIS-Oberfläche selbst. Frank hat die Statusmeldungen der Tour H/42375 (Fahrzeug OD-TS-8050, 02.03.) als Screenshot bereitgestellt und dieselbe Tour steckt mit denselben Stationen und Zeiten in den PRC-Daten. Für die zweite Station stimmen die in CIS angezeigten Klartexte mit den PRC-Codes und Uhrzeiten überein. Code 11 um 08:06 zeigt CIS als Anfahrt, Code 12 um 08:17 als Ankunft, Code 13 um 09:31 als Beginn Laden/Entladen, Code 14 um 09:46 als Ende Laden/Entladen und Code 15 um 09:46 als Abfahrt. Die Bedeutung der Codes stammt damit aus der CIS-Anzeige. Das Mapping-XML behalten wir weiter als Nebenquelle. Es zeigt, welche SPEDION-Formulare auf welchen Code abbilden und welche auftragsart-spezifische Formulierung dahintersteht. Im Katalog stehen beide nebeneinander, damit die Abweichung nachvollziehbar bleibt. Zuerst sehen wir mit value_counts, welche Codes je Meldungstyp überhaupt vorkommen.

In [42]:
# Welche Status-Codes kommen je Meldungstyp tatsächlich in den Daten vor?
prc.groupby("msg_typ")["Status"].value_counts(dropna=False).rename("anzahl").reset_index()

,msg_typ,Status,anzahl
0,Ereignis,<NA>,2
1,FMSStatus,<NA>,6676
2,Position,<NA>,11189
3,Sendposstatus,11,1548
4,Sendposstatus,12,1374
5,Stationsstatus,11,1497
6,Stationsstatus,12,1478
7,Stationsstatus,13,1471
8,Stationsstatus,14,1460
9,Stationsstatus,15,1459


## Katalog und Übersetzung

Der Katalog stellt jedem Code seine C-LOGISTIC-Norm-Bedeutung als Label, die XML-Formulierungen und die Anzahl in den Daten gegenüber. So ist belegt, womit wir übersetzen, die Abweichung zwischen XML und Norm bleibt sichtbar, und die im XML konfigurierten, aber nicht genutzten Codes (43, 44, Tour 5) erscheinen mit n_daten gleich null. Zwei Prüfungen sichern ab, dass jedes Label im XML konfiguriert ist und jeder in den Daten genutzte Code ein Label hat.

In [43]:
import xml.etree.ElementTree as ET

XSI = "{http://www.w3.org/2001/XMLSchema-instance}type"
FORMATTER = {"StationsstatusFormatter": "Stationsstatus",
             "SendposstatusFormatter": "Sendposstatus",
             "TourstatusFormatter":    "Tourstatus"}

# XML als Nebenquelle: je (Meldungstyp, Status) die auftragsart-spezifischen Formulierungen
xml_rows = [{"msg_typ": FORMATTER[f.get(XSI)], "status": int(f.findtext("Status")),
             "xml_text": (f.findtext("StatusText") or "").strip()}
            for f in ET.parse(DATA / "C-LOGISTIC-Mapping.xml").getroot().iter("AbstractFormatter")
            if f.get(XSI) in FORMATTER]
xml_agg = (pd.DataFrame(xml_rows).groupby(["msg_typ", "status"])["xml_text"]
           .agg(lambda s: " | ".join(sorted({t for t in s if t})))
           .reset_index().rename(columns={"xml_text": "xml_varianten"}))

# Labels nach C-LOGISTIC-Norm, bestätigt durch die CIS-Statusmeldungen
label_df = pd.DataFrame([
    ("Stationsstatus",  3, "Gelöscht"),            ("Stationsstatus", 11, "Anfahrt"),
    ("Stationsstatus", 12, "Ankunft"),             ("Stationsstatus", 13, "Beginn Laden/Entladen"),
    ("Stationsstatus", 14, "Ende Laden/Entladen"), ("Stationsstatus", 15, "Abfahrt"),
    ("Sendposstatus",  11, "Geladen"),             ("Sendposstatus",  12, "Entladen"),
    ("Tourstatus",      4, "Fehler"),              ("Tourstatus",     11, "Gestartet"),
    ("Tourstatus",     14, "Beendet"),             ("Tourstatus",     22, "Empfangen vom Fahrzeug"),
    ("Tourstatus",     28, "Aktualisierung Empfangen"),
], columns=["msg_typ", "status", "label"])

# Vorkommen in den Daten
daten_codes = (prc.loc[prc["Status"].notna()].groupby(["msg_typ", "Status"]).size()
               .rename("n_daten").reset_index())
daten_codes["status"] = daten_codes["Status"].astype(int)
daten_codes = daten_codes.drop(columns="Status")

# Katalog: Norm-Label, XML-Formulierung und Vorkommen nebeneinander (left vom XML zeigt auch ungenutzte Codes)
katalog = (xml_agg.merge(label_df,    on=["msg_typ", "status"], how="left")
                  .merge(daten_codes, on=["msg_typ", "status"], how="left")
                  .sort_values(["msg_typ", "status"]).reset_index(drop=True))
katalog["n_daten"] = katalog["n_daten"].fillna(0).astype(int)

# Prüfungen: jedes Label ist im XML konfiguriert, jeder genutzte Code hat ein Label
label_keys = set(zip(label_df["msg_typ"], label_df["status"]))
xml_keys   = set(zip(xml_agg["msg_typ"],  xml_agg["status"]))
benutzte   = prc.loc[prc["Status"].notna(), ["msg_typ", "Status"]].drop_duplicates()
genutzt    = set(zip(benutzte["msg_typ"], benutzte["Status"].astype(int)))
assert label_keys <= xml_keys, f"Label nicht im XML konfiguriert: {label_keys - xml_keys}"
assert genutzt <= label_keys,  f"Code in Daten ohne Label: {genutzt - label_keys}"

# Übersetzung per Merge; drop davor macht die Zelle wiederholbar
prc = prc.drop(columns="Status_Text", errors="ignore").merge(
    label_df.rename(columns={"status": "Status", "label": "Status_Text"}).astype({"Status": "Int64"}),
    on=["msg_typ", "Status"], how="left")

katalog.rename(columns={"msg_typ": "Meldungstyp", "status": "Status",
                        "label": "Bedeutung", "n_daten": "Vorkommen",
                        "xml_varianten": "XML-Formulierungen"})

,Meldungstyp,Status,XML-Formulierungen,Bedeutung,Vorkommen
0,Sendposstatus,11,Lademengen,Geladen,1548
1,Sendposstatus,12,Entlademengen | Workflow,Entladen,1374
2,Stationsstatus,3,Aufgabe abgelehnt | Beladestelle abgelehnt | Entladestel...,Gelöscht,31
3,Stationsstatus,11,Aufgabe angenommen | Beladestelle angenommen | Entladest...,Anfahrt,1497
4,Stationsstatus,12,Ankunft Aufgabe | Ankunft Beladestelle | Ankunft Entlade...,Ankunft,1478
5,Stationsstatus,13,Beginn Aufgabe | Beginn Beladestelle | Beginn Entladestelle,Beginn Laden/Entladen,1471
6,Stationsstatus,14,Ende Aufgabe | Ende Beladestelle (abgeholt) | Ende Entla...,Ende Laden/Entladen,1460
7,Stationsstatus,15,Abfahrt erledigt (Aufgabe) | Abfahrt erledigt (Beladeste...,Abfahrt,1459
8,Stationsstatus,43,ETA Info Ankunftszeit,NaN,0
9,Stationsstatus,44,ETA Info Ankunftszeit mit Lenk- und Ruhezeit,NaN,0


In [44]:
# Beleg an der Tour, die Frank als Screenshot bereitgestellt hat mit ihren fünf Statusstufen.
beleg = prc[(prc["StationID"] == "H/42375-TP2") & (prc["LKW_KENNZ"] == "OD-TS-8050")]
beleg[["StationID", "Status", "Status_Text", "Zeitstempel_Fahrzeug"]].sort_values("Zeitstempel_Fahrzeug")

,StationID,Status,Status_Text,Zeitstempel_Fahrzeug
527,H/42375-TP2,11,Anfahrt,2026-03-02 08:06:45
555,H/42375-TP2,12,Ankunft,2026-03-02 08:17:09
729,H/42375-TP2,13,Beginn Laden/Entladen,2026-03-02 09:31:03
749,H/42375-TP2,14,Ende Laden/Entladen,2026-03-02 09:46:56
750,H/42375-TP2,15,Abfahrt,2026-03-02 09:46:56


Die Prüfung bestätigt die Übersetzung an der realen Station. Dieselben fünf Codes tauchen mit denselben Uhrzeiten wie in Franks Screenshot auf. Allgemein durchlaufen die Stationen den Ablauf Anfahrt, Ankunft, Beginn Laden/Entladen, Ende Laden/Entladen, Abfahrt. Die Häufigkeit nimmt über die Stufen leicht ab, von 1.497 Anfahrten auf 1.459 Abfahrten, ein kleiner Teil der Besuche erreicht also nicht alle fünf Stufen. Für die Standzeit zählt davon Ankunft (12) bis Abfahrt (15) oder Beginn Laden (13) bis Ende Laden (14), das wird sich noch zeigen. Anfahrt (11) ist erst der Beginn der Zufahrt und damit kein Ankunftszeitpunkt.

Zwei Codes verdienen eine Anmerkung. Stationscode 3 ist der einzige in den Daten, der in der CIS-Ansicht nicht vorkam. Die Norm führt ihn als Gelöscht, das SPEDION-Formular dahinter (1336) als vom Fahrer abgelehnt oder gelöscht. Die 31 Fälle sind also nicht ausgeführte Stopps, die genaue Anzeige bleibt aber unbestätigt. Tourcode 4 (Fehler) bündelt laut XML mehrere Storno- und Fehlerursachen. Position, FMSStatus und Ereignis tragen keinen Status und bleiben leer.

# 3 – Gemeinsamer Überblick über die Meldungstypen

In der PRC-Konsolidierung haben wir je Meldungstyp eine Hypothese formuliert, welchen Mehrwert er für die Standzeit- und Routenanalyse haben könnte. Bevor wir diese Hypothesen einzeln prüfen, schaffen wir eine gemeinsame Grundlage, die für alle Typen gilt und sich kompakt zusammen zeigen lässt. Dazu sehen wir uns an, welche Information jeder Typ trägt und in welchem zeitlichen Rhythmus er auftritt. Auf diesen beiden Übersichten verorten wir die einzelnen Typen danach und untersuchen ihre Hypothesen vertieft.

## 3.1 Was trägt jeder Meldungstyp?

Der übersetzte Status sagt, was passiert ist (Ankunft, Abfahrt usw.), aber noch nicht wo und in welcher Tour. Diese Informationen stecken in den bisher nur mitgeführten Schlüsselfeldern. Bevor wir die einzelnen Typen untersuchen, sehen wir uns daher an, welche Schlüssel je Typ überhaupt belegt sind und wie ihre Werte aussehen.

In [45]:
print("Schlüssel je Meldungstyp:")
display(prc.groupby("msg_typ")[["TourID", "StationID", "SendposID"]].count())

print("\nBeispielwert je Schlüssel und Meldungstyp:")
display(prc.groupby("msg_typ")[["TourID", "StationID", "SendposID"]].first())

Schlüssel je Meldungstyp:


,TourID,StationID,SendposID
msg_typ,,,
Ereignis,0,0,0
FMSStatus,0,0,0
Position,0,0,0
Sendposstatus,0,0,2922
Stationsstatus,0,7396,0
Tourstatus,1244,0,0



Beispielwert je Schlüssel und Meldungstyp:


,TourID,StationID,SendposID
msg_typ,,,
Ereignis,NaN,NaN,NaN
FMSStatus,NaN,NaN,NaN
Position,NaN,NaN,NaN
Sendposstatus,NaN,NaN,STK/186527-407337
Stationsstatus,NaN,H/42379-TP1,NaN
Tourstatus,H/42380,NaN,NaN


Jeder Status-Typ trägt genau einen Schlüssel. Der Stationsstatus hat eine StationID, der Sendposstatus eine SendposID, der Tourstatus eine TourID und die jeweils anderen bleiben leer. Das bestätigt die Struktur aus der Vorverarbeitung. Die Beispielwerte sind zusammengesetzt, etwa H/42379-TP1, STK/186527-407337 oder H/42380, und tragen den Tour- und Stationsbezug codiert in sich. Um aus einem Status Ankunft ableiten zu können, wo und in welcher Tour er auftrat, müssen wir diese Schlüssel zuerst auseinandernehmen.

## 3.2 In welchem Rhythmus kommt jeder Meldungstyp?

Neben der Struktur unterscheiden sich die Typen darin, in welchen Zeitabständen sie gemeldet werden. Wir messen je Fahrzeug den Abstand zwischen aufeinanderfolgenden Meldungen desselben Typs, über den Zeitstempel der Fahrzeug-Uhr, also wann die Box die Meldung erzeugt hat. Dabei sind drei Kennzahlen zentral, der typische Abstand zwischen den Meldungen als Median, den Abstand am oberen Ende als 90-Prozent-Marke und den Anteil exakt gleichzeitiger Meldungen. Ein getakteter Typ zeigt einen stabilen Median bei moderater Obergrenze, ein ereignisbasierter Typ einen Median nahe null, eine sehr große Obergrenze und viele gleichzeitige Meldungen.

In [46]:
# Abstand zur vorherigen Meldung desselben Typs, je Fahrzeug (in Sekunden, über die Fahrzeug-Uhr)
prc_sort = prc.sort_values(["LKW_KENNZ", "Zeitstempel_Fahrzeug"])
abstand = prc_sort.groupby(["LKW_KENNZ", "msg_typ"])["Zeitstempel_Fahrzeug"].diff().dt.total_seconds()
ab = pd.DataFrame({"msg_typ": prc_sort["msg_typ"], "abstand_s": abstand}).dropna(subset=["abstand_s"])

# Je Typ: typischer Abstand (Median), oberes Ende (90 %) und Anteil gleichzeitiger Meldungen
rhythmus = (prc.groupby("msg_typ").size().rename("Meldungen").to_frame()
            .join(ab.groupby("msg_typ")["abstand_s"].agg(
                **{"Mittlerer Abstand zwischen Meldungen (s)": "median",
                   "Abstand 90 %-Grenze (s)": lambda s: s.quantile(0.9),
                   "gleichzeitig (%)": lambda s: s.eq(0).mean() * 100}).round(0)))
rhythmus

,Meldungen,Mittlerer Abstand zwischen Meldungen (s),Abstand 90 %-Grenze (s),gleichzeitig (%)
msg_typ,,,,
Ereignis,2,NaN,NaN,NaN
FMSStatus,6676,2.0,3340.0,29.0
Position,11189,330.0,2530.0,4.0
Sendposstatus,2922,0.0,8512.0,55.0
Stationsstatus,7396,3.0,3341.0,21.0
Tourstatus,1244,66.0,53916.0,39.0


Nur die Position ist getaktet mit einem Median von 330 Sekunden (5,5 Minuten) und nur 4 Prozent gleichzeitige Meldungen, das ist also die regelmäßige GPS-Spur des Fahrzeugs. Die vier Status-Typen sind dagegen ereignisbasiert und treten in Schüben auf, wenn an einer Station etwas passiert, nicht in einem festem Takt. Stationsstatus und Sendposstatus haben einen Median von 3 beziehungsweise 0 Sekunden bei einem hohen Anteil exakt gleichzeitiger Meldungen (Sendposstatus 55 Prozent, Stationsstatus 21 Prozent). Der Tourstatus liegt mit Median 66 Sekunden und einer 90-Prozent-Marke von über 53.000 Sekunden erwartbar weit auseinander, weil er Start und Ende einer ganzen Tour klammert. FMSStatus läuft mit diesen Schüben mit (Median 2 Sekunden). Ereignis hat mit nur zwei Meldungen keinen ablesbaren Rhythmus und bleibt in der Tabelle leer. 

Damit ist die Arbeitsteilung für die folgenden Kapitel angelegt. Die ereignisbasierten Status-Typen tragen die Stopp-Information für Reihenfolge und Re-Disposition, die getaktete Position-Spur die gefahrene Route.

# 4 – Stationsstatus: Stations-Zyklus und Standzeit

Die Hypothese zum Stationsstatus lautete, dass er den Stations-Zyklus einer Tour von der Anfahrt bis zur Abfahrt abbildet und damit die zentrale Quelle für Standzeiten und Reihenfolge ist. Diese Hypothese prüfen wir hier in mehreren aufeinander aufbauenden Schritten.

## 4.1 Die StationID zerlegen

Der Stationsstatus trägt seinen Tour- und Stopp-Bezug codiert in der StationID. Bevor wir sie zerlegen, sehen wir uns die rohen Werte an, um ihr Muster zu erkennen.

In [47]:
# Rohe StationID-Werte ansehen, um das Muster zu erkennen
st = prc.loc[prc["msg_typ"] == "Stationsstatus", "StationID"]
print("Eindeutige Beispiele:")
print(st.drop_duplicates().head(12).to_string(index=False))
print("\nStationIDs mit Versionierung '_':", st.str.contains("_").sum())

Eindeutige Beispiele:
    H/42379-TP1
H/42378_2-TP1_2
    H/42381-TP1
    H/42377-TP1
    H/42383-TP2
    H/42375-TP1
    H/42380-TP2
    H/42380-TP3
    H/42376-TP2
    H/42376-TP3
    H/42378-TP5
    H/42379-TP4

StationIDs mit Versionierung '_': 1387


Die StationID ist nach dem Muster Tour-Stopp aufgebaut (H/42379-TP1), wobei beide Teile eine unabhängige Versionsnummer tragen können (H/42378_2-TP1_2). Wir trennen die Basis-Tournummer für die spätere Verknüpfung mit den Soll-Daten, den Stopp und die beiden Versionen. Eine Tour-Version steht für eine Re-Disposition der ganzen Tour, eine Stopp-Version für die Änderung eines einzelnen Stopps. Unversionierte Schlüssel zählen wir als Version 1.

In [48]:
sst = prc["msg_typ"] == "Stationsstatus"

# StationID am '-' in Tour-Teil und Stopp-Teil trennen, dann je Teil die optionale Version am '_'
tour_teil = prc.loc[sst, "StationID"].str.split("-").str[0]
stop_teil = prc.loc[sst, "StationID"].str.split("-").str[1]

prc.loc[sst, "tour_nr"]       = tour_teil.str.split("_").str[0]
prc.loc[sst, "stopp"] = stop_teil.str.split("_").str[0].str.removeprefix("TP").astype(int)
prc.loc[sst, "tour_version"]  = tour_teil.str.split("_").str[1].fillna("1").astype(int)
prc.loc[sst, "stopp_version"] = stop_teil.str.split("_").str[1].fillna("1").astype(int)
for c in ["tour_version", "stopp_version"]:
    prc[c] = prc[c].astype("Int64")   # unversionierte Zeilen bleiben sauber als <NA> für andere Typen

print("Eindeutige Basis-Touren:", prc.loc[sst, "tour_nr"].nunique())
print("Tour-Versionen: ", dict(prc.loc[sst, "tour_version"].value_counts().sort_index()))
print("Stopp-Versionen:", dict(prc.loc[sst, "stopp_version"].value_counts().sort_index()))
prc.loc[sst, ["StationID", "tour_nr", "stopp", "tour_version", "stopp_version"]].head()

Eindeutige Basis-Touren: 199
Tour-Versionen:  {np.int64(1): np.int64(6009), np.int64(2): np.int64(1102), np.int64(3): np.int64(225), np.int64(4): np.int64(15), np.int64(5): np.int64(45)}
Stopp-Versionen: {np.int64(1): np.int64(6242), np.int64(2): np.int64(1154)}


,StationID,tour_nr,stopp,tour_version,stopp_version
30,H/42379-TP1,H/42379,1.0,1,1
51,H/42378_2-TP1_2,H/42378,1.0,2,2
52,H/42378_2-TP1_2,H/42378,1.0,2,2
53,H/42378_2-TP1_2,H/42378,1.0,2,2
55,H/42379-TP1,H/42379,1.0,1,1


Die Zerlegung ergibt 199 verschiedene Basis-Tournummern; ob und wie sich diese mit den 203 Soll-Touren verknüpfen lassen, klären wir in Kapitel 7. Versionierte Schlüssel gibt es 1.387, mit Tour-Versionen bis zur Stufe 5 und Stopp-Versionen bis 2. Es liegen also Touren und einzelne Stopps in mehreren Fassungen vor, was wir in Kapitel 5 als Re-Disposition genauer untersuchen.

## 4.2 Eine StationID ist genau ein Besuch

Da wir für die folgenden Auswertungen je StationID gruppieren wollen, ist vorausgesetzt, dass eine StationID genau einen Stationsbesuch bezeichnet. Das sichern wir ab, indem wir prüfen an wie viele Fahrzeuge und Kalendertage jede StationID gebunden ist. Wäre eine StationID an mehrere Fahrzeuge gebunden, würde die Gruppierung verschiedene Besuche vermischen und die Standzeiten verfälschen.

In [49]:
# An wie viele Fahrzeuge und Kalendertage ist eine StationID gebunden?
stat = prc[prc["msg_typ"] == "Stationsstatus"]
g = stat.groupby("StationID")
print("StationIDs mit mehr als einem Fahrzeug:", (g["LKW_KENNZ"].nunique() > 1).sum())
print("StationIDs mit mehr als einem Kalendertag:",
      (g["Zeitstempel_Fahrzeug"].apply(lambda s: s.dt.date.nunique()) > 1).sum())

StationIDs mit mehr als einem Fahrzeug: 0
StationIDs mit mehr als einem Kalendertag: 7


Keine StationID ist an mehr als ein Fahrzeug gebunden, die Gruppierung mischt also nie zwei Fahrzeuge in einen Besuch. Sieben StationIDs erstrecken sich über zwei Kalendertage; die sehen wir uns an, weil eine über mehrere Tage gespannte StationID die Verweildauer stark verlängern würde.

In [50]:
# Spanne zwischen erster und letzter Meldung der sieben zweitägigen StationsIDs
mehrtags = g["Zeitstempel_Fahrzeug"].apply(lambda s: s.dt.date.nunique()) > 1
detail = (stat[stat["StationID"].isin(mehrtags[mehrtags].index)]
          .groupby("StationID")
          .agg(fahrzeug=("LKW_KENNZ", "first"),
               erste=("Zeitstempel_Fahrzeug", "min"),
               letzte=("Zeitstempel_Fahrzeug", "max")))
detail["spanne_h"] = ((detail["letzte"] - detail["erste"]).dt.total_seconds() / 3600).round(1)
detail.sort_values("spanne_h", ascending=False)

,fahrzeug,erste,letzte,spanne_h
StationID,,,,
H/42468_2-TP6_2,PI-EN-900,2026-03-11 14:24:51,2026-03-16 12:13:11,117.8
H/42578-TP5,OD-TS-8046,2026-03-23 14:11:26,2026-03-26 11:39:43,69.5
H/42571-TP3,PI-EN-900,2026-03-23 14:09:29,2026-03-25 06:04:29,39.9
H/42547-TP5,PI-EN-900,2026-03-19 12:37:11,2026-03-20 15:37:36,27.0
H/42638-TP1,WL-PL-431,2026-03-25 10:43:16,2026-03-26 06:14:03,19.5
H/42506-TP2,OD-TS-8050,2026-03-12 14:41:46,2026-03-13 09:37:20,18.9
H/42663-TP2,WL-PL-431,2026-03-30 18:25:27,2026-03-31 04:20:05,9.9


Die sieben Spannen reichen von rund 10 bis 118 Stunden. Solche Längen scheinen für eine Standzeit an einer einzelnen Station eher ungewöhnlich und deuten auf nachgelagerte Abfahrtsmeldungen hin, bei denen die Abfahrt erst lange nach dem tatsächlichen Wegfahren gesetzt wurde. Für die Standzeit spielt das allerdings keine Rolle, da Spannen dieser Größe später durch die Plausibilitätsgrenze abgefangen werden.

## 4.3 Lebenszyklus und Sammelmeldungs-Prüfung

Wir prüfen je Stationsbesuch, also je StationID, ob alle fünf Stufen vorhanden sind und ob Ankunft und Abfahrt zeitlich auseinanderliegen. Kämen die Stufen gebündelt zur selben Zeit vor, wäre daraus keine Standzeit ablesbar.

In [51]:
stufen = ["Anfahrt", "Ankunft", "Beginn Laden/Entladen", "Ende Laden/Entladen", "Abfahrt"]

# Anzahl vorhandener Lebenszyklus-Stufen je Besuch, gelöschte Stopps haben 0 Stufen
stufen_anzahl = (stat[stat["Status_Text"].isin(stufen)]
                 .groupby("StationID")["Status_Text"].nunique()
                 .reindex(stat["StationID"].unique(), fill_value=0))

print("Stationsbesuche:", len(stufen_anzahl))
print("Verteilung vorhandener Lebenszyklus-Stufen:")
print(stufen_anzahl.value_counts().sort_index())

Stationsbesuche: 1526
Verteilung vorhandener Lebenszyklus-Stufen:
Status_Text
0      31
1      19
2       6
3      11
4       1
5    1458
Name: count, dtype: int64


Die Vollständigkeit bestätigt sich, denn 1.458 der 1.526 Besuche durchlaufen alle fünf Stufen. Die 31 Besuche ohne jede Stufe sind die gelöschten Stopps (Stationscode 3). Die übrigen 37 sind unvollständig, mit einer bis vier Stufen. Die werden im nächsten schritt einma, genauer untersucht. Für die große Mehrheit liegen damit alle fünf Zeitstempel vor und es bleibt zu prüfen, ob und wie weit sie zeitlich auseinanderliegen.

In [52]:
# Die 37 unvollständigen Besuche: welche Stufen fehlen, und wofür sie taugen
stufen = ["Anfahrt", "Ankunft", "Beginn Laden/Entladen", "Ende Laden/Entladen", "Abfahrt"]
vorhanden = stat[stat["Status_Text"].isin(stufen)].groupby("StationID")["Status_Text"].agg(set)
anzahl = vorhanden.apply(len).reindex(stat["StationID"].unique(), fill_value=0)
unvoll = vorhanden.loc[anzahl[(anzahl >= 1) & (anzahl <= 4)].index]

kombination = unvoll.apply(lambda s: " + ".join(x for x in stufen if x in s))
print("Vorhandene Stufen der 37 unvollständigen Besuche:")
print(kombination.value_counts().to_string())
print(f"\nmit Ankunft (für Reihenfolge nutzbar): {unvoll.apply(lambda s: 'Ankunft' in s).sum()}")
print(f"mit Ankunft & Abfahrt (für Standzeit nutzbar): {unvoll.apply(lambda s: {'Ankunft','Abfahrt'} <= s).sum()}")

Vorhandene Stufen der 37 unvollständigen Besuche:
Status_Text
Anfahrt                                                            19
Anfahrt + Ankunft + Beginn Laden/Entladen                          11
Anfahrt + Ankunft                                                   6
Anfahrt + Ankunft + Beginn Laden/Entladen + Ende Laden/Entladen     1

mit Ankunft (für Reihenfolge nutzbar): 18
mit Ankunft & Abfahrt (für Standzeit nutzbar): 0


Die 37 unvollständigen Besuche sind durchweg abgebrochene Tourzyklen. Jeder beginnt mit der Anfahrt und bricht dann ab. 19 melden nur die Anfahrt, die übrigen kommen bis Ankunft oder Beginn, einer bis Ende, aber keiner bis zur Abfahrt. Keiner der 37 trägt Ankunft und Abfahrt zugleich, damit liefert keiner eine Verweildauer und alle fallen ohne Sonderbehandlung aus der Standzeit. Die 18 mit Ankunft ordnen wir in die Reihenfolge ein, der Stopp wurde erreicht, nur nicht abgeschlossen. Die 19 reinen Anfahrt-Meldungen bleiben überall außen vor, weil ohne Ankunft weder eine Position in der Reihenfolge noch eine Standzeit bestimmbar ist.

## 4.4 Die Besuchstabelle: Standzeit je Stationsbesuch

Hier werden die bisherigen Erkenntnisse einmal zusammengeführt. Eine Zeile je Stationsbesuch, mit Tour, Stopp, den Zeitstempeln und der ablesbaren Standzeit. Wir zeigen unsere beiden Standzeit-Definitionen nebeneinander, die reine Be-/Entladedauer von Beginn bis Ende und die gesamte Verweildauer von Ankunft bis Abfahrt. Danach sortieren wir innerhalb jeder Tour nach der Stopp-Nummer, sodass sich eine Tour wie ein Fahrtverlauf von oben nach unten lesen lässt. Leere Zellen markieren Stufen, die nicht gemeldet wurden, also die abgelehnten oder unvollständigen Besuche, die keine Standzeit hergeben.

In [53]:
# Breitformat: eine Zeile je Stationsbesuch, je Stufe ihr frühester Zeitstempel
zeiten = stat.groupby(["StationID", "Status_Text"])["Zeitstempel_Fahrzeug"].min().unstack()

# Beschreibende Spalten je Besuch; first() ist zulässig, weil 4.2 die Eindeutigkeit belegt hat
besuche = (stat.groupby("StationID")
                .agg(fahrzeug=("LKW_KENNZ", "first"), tour=("tour_nr", "first"), stopp=("stopp", "first"))
                .join(zeiten))

# Zwei Standzeit-Definitionen in Minuten
besuche["be_entlade_min"] = ((besuche["Ende Laden/Entladen"] - besuche["Beginn Laden/Entladen"]).dt.total_seconds() / 60).round(1)
besuche["verweil_min"]    = ((besuche["Abfahrt"] - besuche["Ankunft"]).dt.total_seconds() / 60).round(1)

# Verwertbarkeit je Definition: Sekundenschub (< 1 min, kein echter Aufenthalt) vs verwertbar (>= 5 min)
for sp, name in [("verweil_min", "Verweildauer"), ("be_entlade_min", "Be-/Entladedauer")]:
    s = besuche[sp].dropna(); nutz = s >= 5
    print(f"{name:18}: vorhanden={len(s)}  Schub <1min={(s < 1).mean()*100:.0f}%  "
          f"verwertbar >=5min={nutz.sum()} ({nutz.mean()*100:.0f}%, Median {s[nutz].median():.0f} min)  max={s.max():.0f} min")

# Innerhalb der Tour nach Stopp-Nummer sortieren 
besuche = besuche.sort_values(["tour", "stopp"])

spalten = ["fahrzeug", "tour", "stopp", "Ankunft", "Beginn Laden/Entladen",
           "Ende Laden/Entladen", "Abfahrt", "be_entlade_min", "verweil_min"]
besuche[spalten].rename(columns={
    "fahrzeug": "Fahrzeug", "tour": "Tour", "stopp": "Stopp",
    "be_entlade_min": "Be-/Entladedauer (min)", "verweil_min": "Verweildauer (min)"})

Verweildauer      : vorhanden=1458  Schub <1min=41%  verwertbar >=5min=686 (47%, Median 40 min)  max=2296 min
Be-/Entladedauer  : vorhanden=1459  Schub <1min=48%  verwertbar >=5min=593 (41%, Median 32 min)  max=307 min


,Fahrzeug,Tour,Stopp,Ankunft,Beginn Laden/Entladen,Ende Laden/Entladen,Abfahrt,Be-/Entladedauer (min),Verweildauer (min)
StationID,,,,,,,,,
H/42374-TP2,PLO-TS-856,H/42374,2.0,2026-03-02 07:00:34,2026-03-02 08:05:04,2026-03-02 09:26:01,2026-03-02 09:28:17,81.0,147.7
H/42374-TP4,PLO-TS-856,H/42374,4.0,2026-03-02 11:00:37,2026-03-02 14:59:43,2026-03-02 16:24:25,2026-03-02 16:25:14,84.7,324.6
H/42375-TP1,OD-TS-8050,H/42375,1.0,2026-03-02 05:50:16,2026-03-02 05:50:16,2026-03-02 05:50:17,2026-03-02 05:50:17,0.0,0.0
H/42375_3-TP1_2,OD-TS-8050,H/42375,1.0,2026-03-02 16:08:10,2026-03-02 16:08:11,2026-03-02 16:08:11,2026-03-02 16:08:12,0.0,0.0
H/42375-TP2,OD-TS-8050,H/42375,2.0,2026-03-02 08:17:09,2026-03-02 09:31:03,2026-03-02 09:46:56,2026-03-02 09:46:56,15.9,89.8
...,...,...,...,...,...,...,...,...,...
H/42666_2-TP2_2,OD-TS-8046,H/42666,2.0,2026-03-31 08:06:36,2026-03-31 08:06:41,2026-03-31 08:06:41,2026-03-31 08:06:41,0.0,0.1
H/42666_2-TP3_2,OD-TS-8046,H/42666,3.0,2026-03-31 08:06:45,2026-03-31 08:06:48,2026-03-31 08:06:48,2026-03-31 08:06:49,0.0,0.1
H/42666_2-TP4_2,OD-TS-8046,H/42666,4.0,2026-03-31 09:59:35,2026-03-31 10:30:22,2026-03-31 10:30:23,2026-03-31 10:30:23,0.0,30.8


In [54]:
# Lässt sich der sich über einen anderen Zeitstempel auflösen?
# Die Verweildauer wird dieselbe (Abfahrt - Ankunft), nur die zugrunde liegende Uhr wechselt.
def verweildauer(zeitspalte):
    z = stat.groupby(["StationID", "Status_Text"])[zeitspalte].min().unstack()
    return (z["Abfahrt"] - z["Ankunft"]).dt.total_seconds() / 60

print("Schub-Anteil (<1 min) je Zeitstempel der Stationsmeldung:")
for sp in ["Zeitstempel_Fahrzeug", "Zeitstempel_GPS", "Zeitstempel_Server"]:
    v = verweildauer(sp).dropna()
    print(f"  {sp:22}: Schub {(v < 1).mean()*100:.0f}%   verwertbar >=5min {(v >= 5).mean()*100:.0f}%")

# Rettet ein anderer Zeitstempel die Besuche, die über die Fahrzeug-Uhr im Schub liegen?
schub = verweildauer("Zeitstempel_Fahrzeug")
schub_ids = schub[schub < 1].index
v_gps = verweildauer("Zeitstempel_GPS").reindex(schub_ids)
v_srv = verweildauer("Zeitstempel_Server").reindex(schub_ids)
korr = pd.concat([verweildauer("Zeitstempel_Fahrzeug"),
                  verweildauer("Zeitstempel_Server")], axis=1).dropna()
print(f"\nKorrelation Fahrzeug- vs Server-Verweildauer: {korr.iloc[:, 0].corr(korr.iloc[:, 1]):.2f}")
print(f"Von {len(schub_ids)} Schub-Besuchen rettet GPS {(v_gps >= 5).sum()}, "
      f"Server {(v_srv >= 5).sum()} auf eine Spanne >=5min")

Schub-Anteil (<1 min) je Zeitstempel der Stationsmeldung:
  Zeitstempel_Fahrzeug  : Schub 42%   verwertbar >=5min 47%
  Zeitstempel_GPS       : Schub 42%   verwertbar >=5min 47%
  Zeitstempel_Server    : Schub 41%   verwertbar >=5min 47%

Korrelation Fahrzeug- vs Server-Verweildauer: 0.98
Von 607 Schub-Besuchen rettet GPS 0, Server 1 auf eine Spanne >=5min


Die beiden Definitionen ergänzen sich. Die Be-/Entladedauer von Beginn bis Ende bleibt nach oben gutartig (höchstens rund 5 Stunden), ist aber bei 48 Prozent der Besuche unter einer Minute, weil Ladebeginn und -ende oft im selben Moment gemeldet werden. Die Verweildauer von Ankunft bis Abfahrt erfasst etwas häufiger eine echte Spanne (47 Prozent der Besuche mindestens 5 Minuten, Median 40 Minuten), wird aber bei den sieben verschleppten Abfahrten unrealistisch groß, bis rund 2.300 Minuten und damit etwa 38 Stunden. Dass keine Spalte negative Werte enthält, belegt die korrekte zeitliche Reihenfolge der Stufen. Für die weitere Analyse wird die Verweildauer von Ankunft bis Abfahrt verwendet, weil sie die gesamte Standzeit am Kunden erfasst und etwas mehr verwertbare Besuche liefert als die reine Be-/Entladedauer.

Der entscheidende Punkt steht in den rund 42 Prozent, deren Verweildauer unter einer Minute liegt. Bei diesen Besuchen wird der ganze Stations-Zyklus in einem Sekundenschub gemeldet, sodass keine echte Standzeit übrig bleibt. Die Prüfung darüber zeigt, dass sich dieser Schub über keinen der drei Zeitstempel der Meldung auflösen lässt. Über die Fahrzeug-, die GPS- und die Server-Uhr liegt der Schub-Anteil gleichermaßen bei rund 42 Prozen. Die Fahrzeug- und die Server-Verweildauer korrelieren zu 0,98 und von den gut 600 Schub-Besuchen rettet kein GPS- und nur ein einziger Server-Zeitstempel eine Spanne von mindestens fünf Minuten. Die Box reicht die Statusstufen offenbar gebündelt zu einem Zeitpunkt nach, statt sie im Verlauf des Stopps einzeln abzusetzen.

Für rund die Hälfte der Besuche liefert der Stationsstatus eine verwertbare Standzeit, für die andere Hälfte gibt er sie grundsätzlich nicht her, unabhängig von der gewählten Uhr. Für diese Hälfte muss die Standzeit aus einer anderen Quelle kommen. Welche das sein wird, hängt davon ab, wie dicht die verfügbaren GPS-Spuren während eines Stopps sind. Das ist an dieser Stelle noch offen und wird im weiteren Verlauf geprüft.

## Zwischenfazit Stationsstatus

Der Stationsstatus ist für beide Projektziele die zentrale Ist-Quelle, und seine innere Aussagekraft ist belegt, mit einer klaren Grenze bei der Standzeit.
1.458 der 1.526 Stationsbesuche, also 96 Prozent, durchlaufen den vollständigen Lebenszyklus. Eine verwertbare Standzeit von mindestens fünf Minuten lässt sich daraus aber nur für rund die Hälfte ablesen. Die andere Hälfte meldet den Zyklus in einem Sekundenschub, der sich über keinen der drei Zeitstempel der Meldung auflösen lässt. 

Aus den Ankunftszeiten lässt sich die gefahrene Reihenfolge je Tour rekonstruieren. Wie stark sie von der geplanten Stopp-Nummer abweicht und wie viele Touren re-disponiert wurden, untersuchen wir ebenfalls im nächsten Schritt.

Belegt ist die innere Aussagekraft, also dass die Daten Standzeit für die verwertbare Hälfte und Reihenfolge hergeben. Ob die abgelesenen Standzeiten der realen Verweildauer entsprechen, entscheidet ein späterer Abgleich. Bei der Reihenfolge bleibt noch offen, wie eng sich die gefahrenen Stopps den geplanten zuordnen lassen.

# 5 – Reihenfolge und Re-Disposition aus dem Stationsstatus

Aus derselben Grundlage wie die Standzeit, also Tour, Stopp und Ankunftszeit je Besuch, lesen wir hier zwei Dinge ab, die nur im PRC-Stationsstatus stecken. Die Reihenfolge, in der die Stopps angefahren wurden, und die Re-Disposition, also nachträglich geänderte Touren. Beide Auswertungen sind PRC-intern. Wir vergleichen hier noch nicht mit dem CIS-Plan, weil PRC keine Plandaten enthält, sondern prüfen die gefahrene Zeitfolge gegen die PRC-eigene Stopp-Nummerierung. Den Bezug zum tatsächlichen Soll-Plan stellen wir erst später her.

## Kodiert die Stopp-Nummer die geplante Reihenfolge?

Bevor wir in die PRC-interne Auswertung gehen, sichern wir eine Eigenschaft der Soll-Daten ab, die wir später brauchen werden. Dort vergleichen wir die gefahrene Reihenfolge gegen die NUMMER-Spalte aus den CIS-Daten. Das setzt voraus, dass diese NUMMER überhaupt die geplante Abfolge kodiert und nicht nur eine beliebige Kennung ist. Wir sortieren die Stopps jeder Tour einmal nach NUMMER und einmal nach geplanter Ankunftszeit. Stimmen beide Reihenfolgen überein, dann gibt die NUMMER die geplante Abfolge wieder.

In [55]:
# Kodiert die Soll-NUMMER überhaupt die geplante Reihenfolge?
gleich = 0
for tour, g in soll.groupby("TOURNR"):
    g = g.dropna(subset=["ANKUNFT"])
    if len(g) < 2:
        continue
    if g.sort_values("NUMMER")["NUMMER"].tolist() == g.sort_values("ANKUNFT")["NUMMER"].tolist():
        gleich += 1
print(f"Touren, in denen die NUMMER-Reihenfolge der geplanten Ankunftsreihenfolge entspricht: {gleich} von {soll['TOURNR'].nunique()}")

Touren, in denen die NUMMER-Reihenfolge der geplanten Ankunftsreihenfolge entspricht: 203 von 203


In [56]:
# Ankunftszeit je Besuch und die für die Reihenfolge nötigen Felder
ankunft = stat[stat["Status_Text"] == "Ankunft"].groupby("StationID")["Zeitstempel_Fahrzeug"].min()
besuch = (stat.groupby("StationID")
               .agg(tour=("tour_nr", "first"), version=("tour_version", "first"),
                    stopp=("stopp", "first"))
               .join(ankunft.rename("ankunft")))

# Saubere Vergleichsbasis: Basis-Version, mit Ankunft, Touren mit mindestens zwei Stopps
basis = besuch[(besuch["version"] == 1) & besuch["ankunft"].notna()]
mehr  = basis.groupby("tour").filter(lambda g: len(g) >= 2)

# Sortierung ist nur eindeutig, wenn keine zwei Stopps einer Tour dieselbe Ankunft tragen
mehrdeutig = mehr.groupby("tour")["ankunft"].apply(lambda s: s.duplicated().any()).sum()
print(f"Touren mit mehrdeutiger Ankunftszeit (gleiche Zeit an zwei Stopps): {mehrdeutig}")

# Geplante (Stopp-Nummer) und gefahrene (Ankunftszeit) Reihenfolge je Tour
geplant  = mehr.sort_values(["tour", "stopp"]).groupby("tour")["stopp"].apply(list)
gefahren = mehr.sort_values(["tour", "ankunft"]).groupby("tour")["stopp"].apply(list)

# „nach_nummer": Stopps sortiert nach PRC-TP-Nummer; „nach_ankunft": sortiert nach Ankunftszeit.
# Beide stammen aus PRC – verglichen wird die PRC-eigene Nummerierung gegen die gefahrene Zeitfolge.
touren = pd.DataFrame({"nach_nummer": geplant, "nach_ankunft": gefahren})
touren["n_stopps"]   = touren["nach_nummer"].apply(len)
touren["abweichung"] = touren["nach_nummer"] != touren["nach_ankunft"]
touren["nach_nummer"]  = touren["nach_nummer"].apply(lambda l: "→".join(f"TP{int(n)}" for n in l))
touren["nach_ankunft"] = touren["nach_ankunft"].apply(lambda l: "→".join(f"TP{int(n)}" for n in l))

print(f"Touren mit >=2 vollständigen Stopps: {len(touren)} | mit abweichender Ist-Reihenfolge: "
      f"{touren['abweichung'].sum()} ({touren['abweichung'].mean()*100:.0f}%)")
touren.sort_values("abweichung", ascending=False)[["n_stopps", "nach_nummer", "nach_ankunft", "abweichung"]]

Touren mit mehrdeutiger Ankunftszeit (gleiche Zeit an zwei Stopps): 0
Touren mit >=2 vollständigen Stopps: 185 | mit abweichender Ist-Reihenfolge: 57 (31%)


,n_stopps,nach_nummer,nach_ankunft,abweichung
tour,,,,
H/42547,7,TP1→TP2→TP3→TP4→TP5→TP6→TP7,TP2→TP1→TP3→TP4→TP6→TP5→TP7,True
H/42526,11,TP1→TP2→TP3→TP4→TP5→TP6→TP7→TP8→TP9→TP10→TP11,TP1→TP2→TP3→TP4→TP5→TP9→TP6→TP7→TP8→TP10→TP11,True
H/42525,13,TP1→TP2→TP3→TP4→TP5→TP6→TP7→TP8→TP9→TP10→TP11→TP12→TP15,TP1→TP2→TP3→TP4→TP5→TP6→TP9→TP10→TP7→TP8→TP11→TP12→TP15,True
H/42555,10,TP2→TP3→TP4→TP5→TP6→TP7→TP8→TP9→TP10→TP11,TP2→TP3→TP4→TP9→TP8→TP6→TP7→TP5→TP10→TP11,True
H/42496,5,TP2→TP3→TP4→TP5→TP6,TP3→TP2→TP6→TP4→TP5,True
...,...,...,...,...
H/42497,9,TP2→TP3→TP4→TP5→TP6→TP8→TP9→TP10→TP11,TP2→TP3→TP4→TP5→TP6→TP8→TP9→TP10→TP11,False
H/42499,7,TP2→TP3→TP4→TP5→TP6→TP7→TP8,TP2→TP3→TP4→TP5→TP6→TP7→TP8,False
H/42500,8,TP2→TP3→TP4→TP5→TP6→TP7→TP8→TP9,TP2→TP3→TP4→TP5→TP6→TP7→TP8→TP9,False


Von den 199 Basis-Touren bleiben für diesen Vergleich 185 übrig. Eine Reihenfolge lässt sich nur dort bilden, wo mindestens zwei Stopps der ursprünglichen Tourfassung eine Ankunft gemeldet haben. Die übrigen vierzehn Touren tragen weniger als zwei auswertbare Stopps. Bei 57 dieser 185 Touren (31 Prozent) weicht die nach Ankunftszeit gefahrene Reihenfolge von der PRC-Stopp-Nummer ab, etwa bei H/42445 mit der Nummern-Folge TP1→TP3→TP4 gegen die gefahrene Folge TP3→TP4→TP1.

Beide verglichenen Spalten stammen aus den PRC-Daten. Die eine sortiert die Stopps nach ihrer TP-Nummer, die andere nach der tatsächlichen Ankunftszeit. Verglichen wird also nicht mit dem Plan, sondern die PRC-eigene Nummerierung gegen die gefahrene Zeitfolge. Wir ordnen nach der Ankunft (Code 12, dem Erreichen des Stopps), nicht nach der Anfahrt (Code 11). Der Sekundenschub stört hier nicht, denn er betrifft die Stufen innerhalb eines Stopps, während die Ankunftszeiten verschiedener Stopps eindeutig auseinanderliegen.

Diese PRC-interne Reihenfolge ist ausdrücklich eine Vorstufe, keine Aussage über die Plan-Abweichung. Sie trägt nur, solange die PRC-Nummer der Soll-NUMMER entspricht und genau das scheitert an den Re-Dispositionen, wo PRC nach der Ausführung neu nummeriert. Die 57 sind deshalb eher die Untergrenze, die die geplanten Umstellungen nicht erfasst. Die tatsächliche Abweichung vom geplanten Soll-Ablauf wird im weiteren Verlauf noch eingehender behandelt.

## Re-Disposition: während der Fahrt geänderte Touren

Im ersten Teil haben wir re-disponierte Stopps ausgeklammert, um eine saubere Basis-Aussage zu bekommen. Sie sind aber selbst ein Reihenfolge-Signal, welches ausschließlich in den PRC-Stationsdaten steckt, weil die Versionierung nur in der StationID auftaucht. Wir sehen uns an, wie viele Touren betroffen sind und wie sich eine solche Änderung im Zeitverlauf darstellt.

In [57]:
# Version und Ankunft je Stationsbesuch
ankunft = stat[stat["Status_Text"] == "Ankunft"].groupby("StationID")["Zeitstempel_Fahrzeug"].min()
versionen = (stat.groupby("StationID")
                  .agg(tour=("tour_nr", "first"), version=("tour_version", "first"), stopp=("stopp", "first"))
                  .join(ankunft.rename("ankunft")))

betroffen = versionen[versionen["version"] > 1]["tour"].nunique()
print(f"Touren mit re-disponierten Stopps (Version > 1): {betroffen} von {versionen['tour'].nunique()}")
max_ver = versionen.groupby("tour")["version"].max()
print("Höchste Version je betroffener Tour:")
print(max_ver[max_ver > 1].value_counts().sort_index().rename_axis("version").to_string())

# Eine betroffene Tour im Zeitverlauf – Originalfassung und spätere Änderung
print("\nBeispiel-Tour H/42379 – Stopps nach Ankunft:")
versionen[versionen["tour"] == "H/42379"].sort_values("ankunft")[["version", "stopp", "ankunft"]]

Touren mit re-disponierten Stopps (Version > 1): 71 von 199
Höchste Version je betroffener Tour:
version
2    53
3    16
5     2

Beispiel-Tour H/42379 – Stopps nach Ankunft:


,version,stopp,ankunft
StationID,,,
H/42379-TP1,1,1.0,2026-03-02 04:56:48
H/42379-TP4,1,4.0,2026-03-02 06:41:10
H/42379-TP2,1,2.0,2026-03-02 07:15:59
H/42379-TP3,1,3.0,2026-03-02 07:36:37
H/42379-TP5,1,5.0,2026-03-02 08:24:45
H/42379-TP6,1,6.0,2026-03-02 09:16:15
H/42379-TP7,1,7.0,2026-03-02 09:46:08
H/42379_3-TP2_2,3,2.0,2026-03-02 10:01:39
H/42379_3-TP9_2,3,9.0,2026-03-02 10:54:15


71 der 199 Touren tragen re-disponierte Stopps. Tour H/42379 zeigt beispielhaft, dass die sieben Stopps der Originalfassung (Version 1, bis 09:46) folgen ab 10:01 drei Stopps der Version 3, zuerst TP2 erneut, dann TP9 und TP10. Während der Durchführung wurde die Tour also um zusätzliche Stopps erweitert und durch die Versionierung in der StationID auf Stopp-Ebene festgehalten. Das deckt sich mit der Auskunft von HTS, wonach Touren im Planungssystem nachträglich geändert werden und sich dabei die Stopps in Reihenfolge oder Anzahl verschieben, während die bereits abgearbeiteten Stopps unberührt bleiben. PRC macht diese Änderung bis auf den einzelnen Stopp sichtbar, also welcher Stopp in welcher Fassung hinzukam. Der Tourstatus meldet zwar, dass eine Aktualisierung empfangen wurde (Code 28). Ob es sich dabei aber um eine geplante Re-Disposition handelt und was genau sich gegenüber dem ursprünglichen Plan geändert hat, zeigt erst der Soll-Abgleich. Aus PRC allein sehen wir nur, dass eine höhere Version später hinzukam.

In [58]:
# Hängen die beiden Reihenfolge-Signale zusammen? Schnittmenge der abweichenden
# mit den re-disponierten Touren. Beide Mengen stammen aus den Analysen oben.
abweichung_touren = set(touren[touren["abweichung"]].index)
redisp_touren = set(versionen[versionen["version"] > 1]["tour"].unique())
gemeinsam = abweichung_touren & redisp_touren

print(f"Touren mit abweichender Reihenfolge: {len(abweichung_touren)}")
print(f"Re-disponierte Touren:               {len(redisp_touren)}")
print(f"In beiden:                           {len(gemeinsam)}")
print(f"Nur Abweichung, nicht re-disponiert: {len(abweichung_touren - redisp_touren)}")

Touren mit abweichender Reihenfolge: 57
Re-disponierte Touren:               71
In beiden:                           14
Nur Abweichung, nicht re-disponiert: 43


## Hängen die beiden Reihenfolge-Signale zusammen?

Die abweichende Reihenfolge und die Re-Disposition sind nach der Prüfung offenbar zwei verschiedene Signale Die Schnittmenge zeigt, dass sie nicht dasselbe messen. Von den 57 Touren mit abweichender Reihenfolge sind nur 14 zugleich re-disponiert. Die übrigen 43 weichen ab, ohne dass die Tour je geändert wurde. Umgekehrt zeigen 57 der 71 re-disponierten Touren in ihrer Basis-Version keine Reihenfolge-Abweichung. Die Reihenfolge-Abweichung ist damit weit überwiegend ein eigenständiges Phänomen und nicht bloß die Folge einer Re-Disposition. 

Die beiden Analysen beruhen deshalb auf unterschiedlichen Grundgesamtheiten und das folgt aus der jeweiligen Frage. Die Reihenfolge lässt sich nur dort vergleichen, wo mindestens zwei Stopps der ursprünglichen Fassung eine Ankunft tragen, also bei 185 Touren. Die Re-Disposition erfordert nur eine höhere Version an irgendeinem Stopp und lässt sich über alle 199 Touren bestimmen.

# 6 – Position: Dichte der GPS-Spur

Die Position trägt keinen Status und keinen Tour- oder Stoppbezug, sondern bildet die reine GPS-Spur des Fahrzeugs ab. Die Hypothese für diesen Message-Typ lautete, dass sie lückenlos genug ist, um als Grundlage für Geofencing und Standzeit zu dienen. Wir prüfen das über die Abstände zwischen aufeinanderfolgenden Punkten je Fahrzeug und zwar ausschließlich aus echten Position-Meldungen, ohne die GPS-Punkte des FMSStatus, damit alle Fahrzeuge nach demselben Maßstab beurteilt werden. Die Vergleichbarkeit ginge sonst verloren, da drei Fahrzeuge der Fahrzeug-Flotte keinen FMSStatus senden.

In [59]:
pos = prc[prc["msg_typ"] == "Position"].sort_values(["LKW_KENNZ", "Zeitstempel_Fahrzeug"])

# Abstand zwischen aufeinanderfolgenden Positionspunkten je Fahrzeug in Minuten
luecke = pos.groupby("LKW_KENNZ")["Zeitstempel_Fahrzeug"].diff().dt.total_seconds() / 60

dichte = pd.DataFrame({
    "median_min": luecke.groupby(pos["LKW_KENNZ"]).median().round(1),
    "p90_min":    luecke.groupby(pos["LKW_KENNZ"]).quantile(0.90).round(1),
    "p95_min":    luecke.groupby(pos["LKW_KENNZ"]).quantile(0.95).round(1),
    "max_h":     (luecke.groupby(pos["LKW_KENNZ"]).max() / 60).round(1),
})
dichte.sort_values("p95_min").rename(columns={
    "median_min": "Mittlere Lücke (min)",
    "p90_min": "Lücke 90 %-Grenze (min)",
    "p95_min": "Lücke 95 %-Grenze (min)",
    "max_h": "größte Lücke (h)"})

,Mittlere Lücke (min),Lücke 90 %-Grenze (min),Lücke 95 %-Grenze (min),größte Lücke (h)
LKW_KENNZ,,,,
PLO-TS-857,5.4,10.2,15.1,65.4
OD-TS-8044,5.5,10.4,19.7,68.3
OD-TS-8041,5.4,13.1,23.1,112.8
PLO-TS-853,5.4,9.9,27.0,69.0
OD-TS-8048,5.6,17.7,34.2,159.2
WL-PL-431,5.6,50.8,263.1,55.8
OD-TS-8050,5.6,73.8,359.9,54.0
OD-TS-8046,5.5,83.6,360.0,6.0
PI-EN-900,5.8,360.0,360.0,6.0


In [60]:
# Verteilung aller Lücken in Bändern: dichter Betrieb, mittlerer Bereich, Ruhe-Ping
baender = pd.cut(luecke.dropna(), [0, 10, 20, 30, 60, 120, 240, 360, 1e9],
                 labels=["0–10", "10–20", "20–30", "30–60", "60–120", "120–240", "240–360", ">360"])
verteilung = baender.value_counts().sort_index()
print("Positions-Lücken in Minuten-Bändern:")
print(verteilung.to_string())
print(f"\nAnteil unter 10 min: {verteilung['0–10'] / verteilung.sum() * 100:.0f} %")

Positions-Lücken in Minuten-Bändern:
Zeitstempel_Fahrzeug
0–10       8355
10–20       739
20–30       275
30–60       404
60–120      222
120–240      93
240–360     541
>360         87

Anteil unter 10 min: 78 %


In [61]:
# Warum schweigt die Spur im Stand? Die Position-Meldung ist bewegungsgetrieben:
# fast alle Punkte entstehen während der Fahrt. EMR pingt dagegen auch im Stillstand.
print("PRC-Position – Geschwindigkeit der Punkte:")
print(f"  bei 0 km/h (Stand): {(pos['Geschwindigkeit'] == 0).mean()*100:.0f} %")
print(f"  bei >0 km/h (Fahrt): {(pos['Geschwindigkeit'] > 0).mean()*100:.0f} %   Median {pos['Geschwindigkeit'].median():.0f} km/h")
print("\nEMR-Spur – Geschwindigkeit der Punkte:")
print(f"  bei 0 km/h (Stand): {(emr['Geschwindigkeit'] == 0).mean()*100:.0f} %")
print(f"  bei >0 km/h (Fahrt): {(emr['Geschwindigkeit'] > 0).mean()*100:.0f} %")

PRC-Position – Geschwindigkeit der Punkte:
  bei 0 km/h (Stand): 8 %
  bei >0 km/h (Fahrt): 92 %   Median 26 km/h

EMR-Spur – Geschwindigkeit der Punkte:
  bei 0 km/h (Stand): 34 %
  bei >0 km/h (Fahrt): 66 %


In [62]:
# Bleibt die Spur während eines Stopps dicht? Position-Punkte im Ankunft–Abfahrt-Fenster echter Aufenthalte
echte = besuche[besuche["verweil_min"] >= 5].copy()

def punkte_im_fenster(r):
    p = pos.loc[pos["LKW_KENNZ"] == r["fahrzeug"], "Zeitstempel_Fahrzeug"]
    return ((p >= r["Ankunft"]) & (p <= r["Abfahrt"])).sum()

echte["pos_punkte"] = echte.apply(punkte_im_fenster, axis=1)
print(f"Echte Aufenthalte (Verweildauer >= 5 min): {len(echte)}")
print(f"  Position-Punkte im Stopp-Fenster – Median: {echte['pos_punkte'].median():.0f}, Mittel: {echte['pos_punkte'].mean():.1f}")
print(f"  Aufenthalte ganz ohne Position-Punkt: {(echte['pos_punkte'] == 0).mean()*100:.0f} %")

Echte Aufenthalte (Verweildauer >= 5 min): 686
  Position-Punkte im Stopp-Fenster – Median: 0, Mittel: 1.5
  Aufenthalte ganz ohne Position-Punkt: 57 %


In [63]:
# Gegenprobe EMR: dieselben Stopp-Fenster, aber die EMR-Spur. Das begründet die spätere
# Verknüpfung PRC <-> EMR, denn EMR muss genau dort Punkte liefern, wo die Position schweigt.
def emr_im_fenster(r):
    m = emr.loc[emr["LKW_KENNZ"] == r["fahrzeug"], "Meldungszeit"]
    return ((m >= r["Ankunft"]) & (m <= r["Abfahrt"])).sum()

echte["emr_punkte"] = echte.apply(emr_im_fenster, axis=1)
print(f"EMR-Punkte im selben Stopp-Fenster – Median: {echte['emr_punkte'].median():.0f}, "
      f"Mittel: {echte['emr_punkte'].mean():.1f}")
print(f"  Aufenthalte ohne EMR-Punkt: {(echte['emr_punkte'] == 0).mean()*100:.0f} %\n")
echte[["fahrzeug", "verweil_min", "pos_punkte", "emr_punkte"]].head(8)

EMR-Punkte im selben Stopp-Fenster – Median: 26, Mittel: 41.0
  Aufenthalte ohne EMR-Punkt: 0 %



,fahrzeug,verweil_min,pos_punkte,emr_punkte
StationID,,,,
H/42374-TP2,PLO-TS-856,147.7,3,112
H/42374-TP4,PLO-TS-856,324.6,7,123
H/42375-TP2,OD-TS-8050,89.8,0,107
H/42375-TP3,OD-TS-8050,207.8,13,243
H/42375_3-TP6_2,OD-TS-8050,28.0,0,48
H/42376-TP3,PLO-TS-859,72.1,0,20
H/42376-TP4,PLO-TS-859,6.3,0,13
H/42376-TP5,PLO-TS-859,50.9,0,21


Der Grundtakt ist bei allen zwölf Fahrzeugen gleich, mit einem Median von rund 5,5 Minuten und 78 Prozent aller Lücken liegen unter zehn Minuten. Im Fahren ist die Spur also dicht, an den oberen Perzentilen zeigen sich aber zwei Arten von Ruhe-Verhalten. Sieben Fahrzeuge schalten im Stand auf einen Ping alle sechs Stunden, erkennbar daran, dass schon ihr 95-Prozent-Wert bei 360 Minuten liegt. Die übrigen fünf haben stattdessen einzelne sehr lange Lücken über Nacht, sichtbar an den hohen Maximalwerten von bis zu 159 Stunden. 

Entscheidend ist, was während eines echten Stopps passiert. Im Ankunft-Abfahrt-Fenster der Aufenthalte bleiben 57 Prozent ganz ohne Position-Punkt, der Median liegt bei null. Der Grund dafür steht in der Geschwindigkeit der Punkte. 92 Prozent aller Position-Meldungen entstehen bei einer Geschwindigkeit über null, also während der Fahrt und nur 8 Prozent im Stand. Die Box ist damit bewegungsgetrieben, sie pingt beim Fahren und geht im Stand still. Die EMR-Spur verhält sich gegenteilig, denn dort entstehen 34 Prozent der Punkte im Stand. Das heißt sie pingt zeitgetrieben weiter und liegt in denselben Stopp-Fenstern im Median bei 26 Punkten, bei keinem einzigen Aufenthalt bei null. Dieser Anteil entsteht aber nicht durch technische Geräte-Meldungen wie Reboot oder Startup, da solche Meldungen nur rund anderthalb Prozent der EMR-Daten ausmachen. Ohne sie bliebe der Anteil der Stand-Daten nur bei rund einem Drittel.

Damit ist die Hypothese geprüft und leider nur zur Hälfte bestätigt. Die Position erfüllt sie als dichte, fahrer-unabhängige Spur des gefahrenen Wegs und ist damit die Grundlage für die Routendarstellung. Für die Standzeit taugt sie dagegen nicht, weil sie genau im Stand schweigt, wo die Standzeit gemessen werden müsste. Eine andere Quelle muss diese Lücke füllen und die EMR-Spur liefert genau dort die Punkte, wo die Position fehlt. Im weiteren Verlauf prüfen wir also, ob sich daraus eine belastbare Standzeit gewinnen lässt. Für die Routenabweichung ergänzen sich beide PRC-Bestandteile bereits. Die Spur zeigt den gefahrenen Weg zwischen den Stopps und der Stationsstatus gibt den Punkten ihre Stopp-Bedeutung.

## Tourstatus und Sendposstatus – geprüft und nicht weiterverwendet

Die Kernergebnisse dieser Ist-Daten sind die tatsächliche Stopp-Reihenfolge, die Standzeit und die gefahrene Route. Alle drei werden durch Stationsstatus und Position getragen. Die übrigen Meldungstypen verwenden wir aus folgenden Gründen nicht:

Der Sendposstatus meldet das Laden und Entladen einer Sendungsposition. Beide Meldungen gehören aber zu verschiedenen Stationen, dem Aufnahme- und dem Abgabeort, die im Median rund zwölf Kilometer auseinanderliegen; die Zeit dazwischen ist Transportzeit, keine Standzeit. Der Sendposstatus kann den Sekundenschub der Standzeit also nicht auflösen, weil er gar keine Verweildauer an einem einzelnen Stopp trägt.

Der Tourstatus klammert eine Tour zeitlich über Start und Ende und meldet mit Code 28 zusätzlich empfangene Aktualisierungen. Beides bringt für die vier Kennzahlen keinen Mehrwert. Die Tour-Klammer wäre nur nötig, um die Spur eines Fahrzeugs in mehrere Touren am selben Tag zu zerschneiden, doch fast jedes Fahrzeug fährt an einem Tag genau eine Tour, und das Dashboard filtert ohnehin je Tour. Der Aktualisierungscode stützt die über die StationID-Versionen erkannten Re-Dispositionen zwar weitgehend, ist aber das gröbere Signal, weil er jede Art von Aktualisierung erfasst, die Versionssuffixe bleiben deshalb die präzisere Quelle.

# 7 – Soll-Ist-Vergleich auf Stopp-Ebene

Bis hier haben wir die PRC-Ist-Daten für sich geprüft. Jetzt verbinden wir sie mit dem Soll-Plan.

Zunächst klären wir die Grundgesamtheit. Von den 203 geplanten Soll-Touren haben vier keine einzige PRC-Stationsmeldung, sodass 199 Touren in PRC sichtbar sind. Umgekehrt gibt es keine PRC-Tour ohne Soll-Entsprechung. Die 199 sind also eine echte Teilmenge der geplanten Touren und die vier Touren ohne Ist-Daten erscheinen im Verbund als reine Plan-Touren.

Naheliegend wäre gewesen, über die Stopp-Nummer zu verbinden, also TPn zu Soll-NUMMER n. Das trägt aber aufgrund der Neu-Versionierung nicht, wie wir bereits festgestellt haben. Bei Re-Disposition nummeriert PRC nach der Ausführung, der Soll nach dem ursprünglichen Plan und die Nummern laufen auseinander.

Wir verbinden deshalb über die Geodaten. Jeden gefahrenen Stopp, erkannt an seiner Ankunft mit Status 12, wird dem nächstgelegenen Soll-Stopp derselben Tour zugeordnet, nach Ankunftszeit sortiert. Der Ort sagt, welcher Kunde es ist, die Zeit trennt Stopps am selben Ort, bspw. das Depot, das morgens und mittags angefahren wird. Liegt ein gefahrener Stopp weiter als 500 Meter von jedem Soll-Stopp weg, gilt er als nicht zuordenbar. Daraus ergeben sich drei Zustände. Treffer, also geplant und geografisch gefahren, nur geplant, also kein gefahrener Stopp in der Nähe, und nicht zuordbar, also gefahren, aber zu keinem Soll-Stopp passend.

In [64]:
# Welche Soll-Touren haben überhaupt PRC-Stationsmeldungen?
# Das klärt die Differenz zwischen den 203 geplanten und den 199 in PRC sichtbaren Touren.
soll_touren = set(soll["TOURNR"].unique())
prc_touren = set(stat["tour_nr"].unique())
ohne_prc = soll_touren - prc_touren
print(f"Soll-Touren: {len(soll_touren)}")
print(f"PRC-Basis-Touren: {len(prc_touren)}")
print(f"Soll-Touren ohne jede PRC-Stationsmeldung: {len(ohne_prc)}  {sorted(ohne_prc)}")
print(f"PRC-Touren ohne Soll-Entsprechung: {len(prc_touren - soll_touren)}")

Soll-Touren: 203
PRC-Basis-Touren: 199
Soll-Touren ohne jede PRC-Stationsmeldung: 4  ['H/42416', 'H/42429', 'H/42449', 'H/42452']
PRC-Touren ohne Soll-Entsprechung: 0


In [65]:
# Sind die vier Touren ohne PRC-Stationsmeldung wirklich nicht gefahren? Gegenprüfung an der EMR-Spur
ohne_prc = ["H/42416", "H/42429", "H/42449", "H/42452"]
for t in ohne_prc:
    s = soll[soll["TOURNR"] == t]
    kz, tag = s["LKW_KENNZ"].iloc[0], s["ANKUNFT"].iloc[0].date()
    em = emr[(emr["LKW_KENNZ"] == kz) & (emr["Meldungszeit"].dt.date == tag)]
    getroffen = sum(
        haversine(st["GEOY"], st["GEOX"], em["Breitengrad"].values, em["Längengrad"].values).min() < 200
        for _, st in s.iterrows() if len(em) > 0)
    print(f"{t}: {len(s)} geplante Stopps, EMR-Spur am Tag {len(em)} Punkte, davon {getroffen} Stopps getroffen (<200 m)")

H/42416: 3 geplante Stopps, EMR-Spur am Tag 146 Punkte, davon 3 Stopps getroffen (<200 m)
H/42429: 3 geplante Stopps, EMR-Spur am Tag 193 Punkte, davon 3 Stopps getroffen (<200 m)
H/42449: 9 geplante Stopps, EMR-Spur am Tag 63 Punkte, davon 2 Stopps getroffen (<200 m)
H/42452: 3 geplante Stopps, EMR-Spur am Tag 282 Punkte, davon 3 Stopps getroffen (<200 m)


Alle vier Touren ohne PRC-Stationsmeldung haben am Tourtag eine dichte EMR-Spur, die sogar die geplanten Stopps trifft. Bei dreien alle, bei H/42449 zwei von neun. Diese Touren wurden also gefahren, nur hat die PRC-Box für sie keinen Stationsstatus gesendet. 

In [66]:
# Warum nicht einfach über die Stopp-Nummer (TPn -> Soll-NUMMER n) verbinden?
# Gegenprobe: Bei der Nummern-Zuordnung, wie weit liegt die gefahrene Ankunftsposition
# vom gleichnamigen Soll-Ort entfernt?
prc_ank = (stat[stat["Status"] == 12].groupby(["tour_nr", "stopp"])
           .agg(prc_lat=("Latitude", "first"), prc_lon=("Longitude", "first")).reset_index())
nummer = prc_ank.merge(soll[["TOURNR", "NUMMER", "GEOX", "GEOY"]],
                       left_on=["tour_nr", "stopp"], right_on=["TOURNR", "NUMMER"], how="inner")
nummer["dist_m"] = haversine(nummer["prc_lat"], nummer["prc_lon"], nummer["GEOY"], nummer["GEOX"])
print(f"Nummern-Zuordnung geprüft an {len(nummer)} Stopps:")
print(f"  innerhalb 300 m (Nummer trifft den Ort): {(nummer['dist_m'] <= 300).mean()*100:.0f} %")
print(f"  weiter als 300 m (Nummer trifft daneben): {(nummer['dist_m'] > 300).mean()*100:.0f} %")

Nummern-Zuordnung geprüft an 1323 Stopps:
  innerhalb 300 m (Nummer trifft den Ort): 65 %
  weiter als 300 m (Nummer trifft daneben): 35 %


In [67]:
GRENZE_M = 500  # weiter entfernt gilt ein gefahrener Stopp als nicht zuordenbar

# Gefahrene Stopps: Ankunftsposition (Status 12) und Ankunftszeit
gefahren = (stat[stat["Status"] == 12]
            .groupby(["tour_nr", "stopp"])
            .agg(prc_lat=("Latitude", "first"), prc_lon=("Longitude", "first"),
                 ist_ankunft=("Zeitstempel_Fahrzeug", "min"))
            .reset_index().rename(columns={"stopp": "prc_stopp"}))

# je Tour: gefahrene Stopps nach Ankunftszeit dem nächsten noch freien Soll-Stopp zuordnen
zeilen = []
min_dist = []  # Minimaldistanz je gefahrenem Stopp, auch wenn über der Grenze – für die Robustheitsprüfung
for tour, g in gefahren.groupby("tour_nr"):
    s = soll[soll["TOURNR"] == tour].sort_values("ANKUNFT")
    frei = list(s.index)
    for _, r in g.sort_values("ist_ankunft").iterrows():
        if frei:
            d = haversine(r["prc_lat"], r["prc_lon"], s.loc[frei, "GEOY"].values, s.loc[frei, "GEOX"].values)
            j = int(d.argmin())
            min_dist.append(d[j])
            if d[j] <= GRENZE_M:
                idx = frei.pop(j)
                zeilen.append((tour, r["prc_stopp"], int(s.loc[idx, "NUMMER"]), round(d[j]), r["ist_ankunft"]))
                continue
        else:
            min_dist.append(np.inf)
        zeilen.append((tour, r["prc_stopp"], np.nan, np.nan, r["ist_ankunft"]))
zuordnung = pd.DataFrame(zeilen, columns=["tour_nr", "prc_stopp", "soll_nummer", "dist_m", "ist_ankunft"])
zugeordnet = zuordnung["soll_nummer"].notna()

# Verbund aufbauen: Soll-Stopps als Basis, zugeordnete gefahrene Stopps daneben
soll_stopp = soll[["TOURNR", "NUMMER", "NAME1", "ANKUNFT", "ABFAHRT"]].rename(
    columns={"TOURNR": "tour_nr", "NUMMER": "soll_nummer", "ANKUNFT": "soll_ankunft", "ABFAHRT": "soll_abfahrt"})
vergleich = soll_stopp.merge(zuordnung[zugeordnet][["tour_nr", "soll_nummer", "prc_stopp", "ist_ankunft", "dist_m"]],
                             on=["tour_nr", "soll_nummer"], how="left")
vergleich["ausfuehrung"] = np.where(vergleich["prc_stopp"].notna(), "Treffer", "nur geplant")
nicht_zu = zuordnung[~zugeordnet][["tour_nr", "prc_stopp", "ist_ankunft"]].assign(ausfuehrung="nicht zuordenbar")
vergleich = pd.concat([vergleich, nicht_zu], ignore_index=True)
vergleich["zeit_abw_min"] = (vergleich["ist_ankunft"] - vergleich["soll_ankunft"]).dt.total_seconds() / 60

# Plausibilitätsgrenze: eine Ankunft über 12 h neben dem Plan ist kein Verzug,
# sondern eine über Tage wiederverwendete Tournummer (Re-Disposition) -> keine Zeitabweichung
vergleich.loc[vergleich["zeit_abw_min"].abs() > 720, "zeit_abw_min"] = np.nan

print(f"Zuordnung: {zugeordnet.sum()} von {len(zuordnung)} gefahrenen Stopps geografisch zugeordnet "
      f"(Median {zuordnung.loc[zugeordnet,'dist_m'].median():.0f} m, 90% unter {zuordnung.loc[zugeordnet,'dist_m'].quantile(.9):.0f} m)")
print(vergleich["ausfuehrung"].value_counts().to_string())
print("\nZeit-Abweichung Ist−Soll-Ankunft (min) über Treffer:")
print(vergleich["zeit_abw_min"].describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())
vergleich[vergleich["tour_nr"] == "H/42378"].sort_values("soll_nummer")[
    ["soll_nummer", "prc_stopp", "ausfuehrung", "NAME1", "dist_m", "soll_ankunft", "ist_ankunft", "zeit_abw_min"]]

Zuordnung: 1127 von 1382 gefahrenen Stopps geografisch zugeordnet (Median 87 m, 90% unter 239 m)
ausfuehrung
Treffer             1127
nur geplant          450
nicht zuordenbar     255

Zeit-Abweichung Ist−Soll-Ankunft (min) über Treffer:
count    1119.0
mean        4.5
std       132.3
min      -698.4
10%      -124.2
25%       -39.7
50%        -0.3
75%        58.1
90%       160.6
max       495.2


,soll_nummer,prc_stopp,ausfuehrung,NAME1,dist_m,soll_ankunft,ist_ankunft,zeit_abw_min
27,1.0,1.0,Treffer,HTH Hanse GmbH,61.0,2026-03-02 05:00:00,2026-03-02 04:51:33,-8.450000
28,2.0,3.0,Treffer,Kälte Klima Nord GbR,23.0,2026-03-02 07:33:00,2026-03-02 10:16:53,163.883333
29,3.0,4.0,Treffer,Klima-Schröder GmbH,352.0,2026-03-02 07:56:00,2026-03-02 10:25:59,149.983333
30,4.0,5.0,Treffer,Hustede Haustechnik,120.0,2026-03-02 09:41:00,2026-03-02 07:45:15,-115.750000
31,5.0,6.0,Treffer,K+J Haustechnik GmbH,13.0,2026-03-02 11:11:00,2026-03-02 11:24:38,13.633333
32,6.0,7.0,Treffer,HTH Hanse GmbH,31.0,2026-03-02 11:48:00,2026-03-02 11:59:49,11.816667


In [68]:
# Ist die 500-m-Grenze robust gewählt? Verteilung der Minimaldistanzen und Zuordnung je Grenze.
md = np.array(min_dist)
endlich = md[np.isfinite(md)]
print("Perzentile der Minimaldistanz gefahrener Stopp zu nächstem Soll-Stopp (m):")
for p in [50, 75, 90, 95]:
    print(f"  {p}%: {np.percentile(endlich, p):.0f}")
print("\nZuordnung je Grenze:")
for g in [300, 400, 500, 600, 700]:
    print(f"  {g} m: {(md <= g).sum()} zugeordnet, {(md > g).sum()} nicht zuordenbar")

Perzentile der Minimaldistanz gefahrener Stopp zu nächstem Soll-Stopp (m):
  50%: 121
  75%: 245
  90%: 1799
  95%: 3564

Zuordnung je Grenze:
  300 m: 1057 zugeordnet, 325 nicht zuordenbar
  400 m: 1122 zugeordnet, 260 nicht zuordenbar
  500 m: 1127 zugeordnet, 255 nicht zuordenbar
  600 m: 1134 zugeordnet, 248 nicht zuordenbar
  700 m: 1143 zugeordnet, 239 nicht zuordenbar


Der geografische Verbund ordnet 1127 der 1382 gefahrenen Stopps eindeutig einem Soll-Stopp zu, im Median nur 87 Meter entfernt und zu 90 Prozent unter 239 Metern. Die Zuordnung ist also eng und belastbar. 255 gefahrene Stopps liegen weiter als 500 Meter von jedem geplanten Stopp weg und gelten als nicht zuordenbar, das sind entweder unbrauchbare GPS-Positionen oder Fahrten abseits des Plans.

Die 500-Meter-Grenze ist nicht willkürlich. Die Verteilung der Minimaldistanzen springt vom 75-Prozent-Wert bei 245 Metern auf 1799 Meter beim 90-Prozent-Wert. Zwischen rund einem Viertelkilometer und knapp zwei Kilometern liegt also fast nichts. Die Grenze fällt in diese Lücke, und ihre genaue Lage ist unkritisch, weil sich die Zuordnung zwischen 400 und 600 Metern nur um zwölf Stopps ändert.

Auf Soll-Seite ergeben sich 1127 Treffer und 450 nur geplante Stopps, zusammen die 1577 Soll-Stopps. Die nur geplanten umfassen die vier Touren, die in PRC ganz fehlen, sowie einzelne Stopps gefahrener Touren ohne bestätigte Ankunft am Plan-Ort.

Über die Treffer sieht man, dass die Zeitabweichung vom deutlich verfrühten bis zum stark verspäteten Eintreffen reicht, mit einem 10- und 90-Perzentil von rund −124 bis +161 Minuten und einem Median nahe null. Das ist eine breite, echte Plan-Ist-Abweichung, denn die Soll-Zeiten sind glatte Planzeiten und die Ist-Zeiten kommen aus den Fahrzeugmeldungen. Ankünfte mehr als zwölf Stunden neben dem Plan werten wir nicht als Verzug, sondern als über Tage wiederverwendete Tournummer an den Basis-Orten. Als Schwelle wird eine Abweichung von zwölf Stunden gesetzt, denn mehr als ein halber Tag Abweichung ist keine Verspätung desselben Stopps mehr.

In [69]:
# Ist die Greedy-Zuordnung robust? Sie gibt jedem gefahrenen Stopp den nächsten noch freien
# Soll-Stopp. Kritisch wäre das nur, wenn mehrere Soll-Stopps gleich nah lägen. Wir messen,
# wie oft die Zuordnung mehrdeutig ist und wie weit die konkurrierenden Soll-Stopps auseinanderliegen.
mehrdeutig = 0
konkurrenz_dist = []  # Abstand zwischen nächstem und zweitnächstem Soll-Stopp, wo beide <= Grenze
for tour, g in gefahren.groupby("tour_nr"):
    s = soll[soll["TOURNR"] == tour]
    if s.empty:
        continue
    for _, r in g.iterrows():
        d = np.sort(haversine(r["prc_lat"], r["prc_lon"], s["GEOY"].values, s["GEOX"].values))
        if len(d) >= 2 and d[0] <= GRENZE_M and d[1] <= GRENZE_M:
            mehrdeutig += 1
            konkurrenz_dist.append(d[1] - d[0])

kd = np.array(konkurrenz_dist)
print(f"Gefahrene Stopps mit mehr als einem Soll-Stopp in {GRENZE_M} m: {mehrdeutig} von {len(gefahren)}")
print(f"Abstand nächster zu zweitnächstem konkurrierenden Soll-Stopp (m):")
for p in [50, 75, 90]:
    print(f"  {p}%: {np.percentile(kd, p):.0f}")
print(f"Konkurrenten am praktisch selben Ort (<50 m auseinander): {(kd < 50).sum()} ({(kd < 50).mean()*100:.0f}%)")

Gefahrene Stopps mit mehr als einem Soll-Stopp in 500 m: 816 von 1382
Abstand nächster zu zweitnächstem konkurrierenden Soll-Stopp (m):
  50%: 0
  75%: 0
  90%: 0
Konkurrenten am praktisch selben Ort (<50 m auseinander): 793 (97%)


Neben der Distanzgrenze ist auch die Zuordnungslogik selbst zu prüfen, denn sie ist greedy. Jeder gefahrene Stopp bekommt in zeitlicher Reihenfolge den nächsten noch freien Soll-Stopp, ohne spätere Zuordnungen vorauszuschauen. Kritisch wäre das, wenn mehrere Soll-Stopps gleich nah lägen und die Wahl dann von der Reihenfolge der Abarbeitung abhinge. Die Prüfung zeigt, dass die Mehrdeutigkeit zwar häufig ist, 816 der 1.382 gefahrenen Stopps haben mehr als einen Soll-Stopp in Reichweite, aber dennoch folgenlos bleibt. Bei 97 Prozent dieser Fälle liegen die konkurrierenden Soll-Stopps weniger als 50 Meter auseinander, der Median ihres Abstands ist null. Es sind also dieselben Orte, die in einer Tour mehrfach als Stopp auftreten, allen voran die Basis.

## 7.1 Ist-Standzeit und Standzeit-Abweichung

Die zweite Abweichungsspalte ist die Standzeit. Wir rechnen sie nicht neu, sondern greifen auf die Besuchstabelle aus Kapitel 4 zurück, die je Stationsbesuch die Be-/Entladedauer und die Verweildauer enthält. Da ein geplanter Stopp mehrfach besucht werden kann, etwa bei re-disponierten Wiederbesuchen, nehmen wir je Stopp den ersten Besuch mit der frühesten Ankunft, passend zur Logik einer Zeile je Stopp. Die geplante Standzeit ergibt sich aus geplanter Ankunft und Abfahrt im Soll und wird mit der tatsächlichen Verweildauer verglichen. Dabei ist zu wissen, dass die Soll-Standzeit eine pauschale Planvorgabe ist. 93 Prozent der Stopps tragen einen von vier Werten, am häufigsten 30 Minuten. Die Standzeit-Abweichung misst also gegen einen Planungsstandard, nicht gegen eine kundenindividuell kalkulierte Erwartung und zeigt damit eher, wie gut die Pauschale die Realität trifft, als dass ein einzelner Halt im engeren Sinn zu lang war.

In [70]:
# Standzeit je Treffer: Verweildauer des gefahrenen Stopps an den Soll-Stopp hängen
erst_besuch = (besuche.reset_index().sort_values("Ankunft")
               .drop_duplicates(["tour", "stopp"], keep="first")
               [["tour", "stopp", "verweil_min"]]
               .rename(columns={"tour": "tour_nr", "stopp": "prc_stopp", "verweil_min": "ist_verweil_min"}))
vergleich = vergleich.merge(erst_besuch, on=["tour_nr", "prc_stopp"], how="left")

vergleich["soll_standzeit_min"] = ((vergleich["soll_abfahrt"] - vergleich["soll_ankunft"]).dt.total_seconds() / 60).round(1)
vergleich["ist_schub"] = vergleich["ist_verweil_min"] < 1
verwertbar = vergleich["ist_verweil_min"] >= 5
vergleich["standzeit_abw_min"] = (vergleich["ist_verweil_min"] - vergleich["soll_standzeit_min"]).round(1)
vergleich.loc[~verwertbar, "standzeit_abw_min"] = pd.NA

beide = vergleich["ist_verweil_min"].notna() & vergleich["soll_standzeit_min"].notna()
print(f"Stopps mit Soll- und Ist-Standzeit: {beide.sum()}")
print(f"  davon Schub <1 min (kein echter Aufenthalt): {int((beide & vergleich['ist_schub']).sum())}")
print(f"  davon verwertbar >=5 min: {int((beide & verwertbar).sum())}")
print(f"  dazwischen (1-5 min): {int((beide & ~vergleich['ist_schub'] & ~verwertbar).sum())}")
print("\nStandzeit-Abweichung Ist−Soll (min, nur verwertbar >=5 min):")
print(vergleich.loc[verwertbar, "standzeit_abw_min"].describe(percentiles=[.1, .5, .9]).round(1).to_string())
vergleich[vergleich["tour_nr"] == "H/42378"].sort_values("soll_nummer")[
    ["soll_nummer", "prc_stopp", "ausfuehrung", "NAME1", "dist_m",
     "soll_ankunft", "ist_ankunft", "zeit_abw_min"]].rename(columns={
    "soll_nummer": "Soll-Stopp", "prc_stopp": "PRC-Stopp", "ausfuehrung": "Ausführung",
    "NAME1": "Kunde", "dist_m": "Distanz (m)",
    "soll_ankunft": "Soll-Ankunft", "ist_ankunft": "Ist-Ankunft", "zeit_abw_min": "Zeitabweichung (min)"})

Stopps mit Soll- und Ist-Standzeit: 1113
  davon Schub <1 min (kein echter Aufenthalt): 392
  davon verwertbar >=5 min: 577
  dazwischen (1-5 min): 144

Standzeit-Abweichung Ist−Soll (min, nur verwertbar >=5 min):
count     577.0
mean       17.9
std        95.6
min      -839.9
10%       -19.4
50%         6.2
90%        69.2
max      1566.0


,Soll-Stopp,PRC-Stopp,Ausführung,Kunde,Distanz (m),Soll-Ankunft,Ist-Ankunft,Zeitabweichung (min)
27,1.0,1.0,Treffer,HTH Hanse GmbH,61.0,2026-03-02 05:00:00,2026-03-02 04:51:33,-8.450000
28,2.0,3.0,Treffer,Kälte Klima Nord GbR,23.0,2026-03-02 07:33:00,2026-03-02 10:16:53,163.883333
29,3.0,4.0,Treffer,Klima-Schröder GmbH,352.0,2026-03-02 07:56:00,2026-03-02 10:25:59,149.983333
30,4.0,5.0,Treffer,Hustede Haustechnik,120.0,2026-03-02 09:41:00,2026-03-02 07:45:15,-115.750000
31,5.0,6.0,Treffer,K+J Haustechnik GmbH,13.0,2026-03-02 11:11:00,2026-03-02 11:24:38,13.633333
32,6.0,7.0,Treffer,HTH Hanse GmbH,31.0,2026-03-02 11:48:00,2026-03-02 11:59:49,11.816667


Für 1.113 Stopps liegt sowohl eine geplante als auch eine tatsächliche Standzeit vor, aber nur ein Teil ist verwertbar. 392 Stopps sind Sekundenschübe unter einer Minute, weitere 144 liegen zwischen einer und fünf Minuten, beide zu kurz für eine echte Standzeit. Nur die 577 mit einer Verweildauer ab fünf Minuten bekommen eine Standzeit-Abweichung, für die übrigen bleibt die Spalte leer. 

Über diese 577 streut die Abweichung ziemlich breit. Das 10- und das 90-Perzentil liegen bei rund −19 und +69 Minuten, der Median bei gut 6 Minuten länger als geplant. Vereinzelt treten extreme Werte bis über tausend Minuten in beide Richtungen auf, auch diese stammen vermutlich aus einzelnen verschleppten oder fehlenden Meldungen und ziehen den Mittelwert unbrauchbar nach oben, weshalb hier der Median die belastbare Kennzahl ist. Die verwertbare Hälfte zeigt damit reale, aber moderate Über- und Unterschreitungen der geplanten Standzeit, überlagert von wenigen Ausreißern aus Datenfehlern.

Tour H/42378 zeigt, warum die Unterscheidung nötig ist. Alle sechs Stopps sind über den geografischen Verbund getroffen, aber nur ein Teil trägt eine Abweichung. Am Depot, Stopp 1 bei HTH Hanse, stand das Fahrzeug real deutlich länger, als der Plan vorsieht, der diesen Stopp nur als Zeitpunkt mit einer Soll-Standzeit von null führt. Die Stopps 3 und 5, Klima-Schröder und K+J Haustechnik, haben mit 5 und 7 Minuten verwertbare, gegenüber der Planung kürzere Standzeiten. Die Stopps 2 und 4, Kälte Klima Nord und Hustede, melden dagegen nur ein bis drei Minuten, zu kurz für eine echte Standzeit, sodass hier die Abweichung leer bleibt. Die geografische Zuordnung ist überall eng, alle Distanzen aus dem Verbund liegen unter wenigen hundert Metern, die Stopp-Nummern in PRC laufen aber gegenüber dem Soll versetzt, also Soll 2 zu PRC 3, Soll 3 zu PRC 4 und so fort. Das ist genau die Verschiebung, die der Nummern-Verbund falsch gepaart hätte.
Damit sind zwei der drei hier behandelten PRC-Abweichungsspalten vollständig gefüllt, Zeit und Reihenfolge, und für die dritte, die Standzeit nur vorläufig mit der verwertbaren Hälfte aus dem Stationsstatus. Für die andere Hälfte, die im Schub steckt, schauen wir uns nochmal die EMR-Spur ergänzend an.

## 7.2 Aggregation auf Tour-Ebene

Die Stopp-Tabelle ist die eigentliche Ist-Tabelle, die Tour-Ebene ist eine Aggregation darüber. Wir verdichten sie zu einer Zeile je Tour, die die Befunde der vorigen Schritte zusammenfasst, ohne neue Annahmen, nur Summen und Zähler über die bereits belegten Stopp-Werte. Die Reihenfolge-Abweichung übernehmen wir unverändert, die Standzeit summieren wir nur über die verwertbaren Stopps.

In [71]:
# Plan-bezogene Reihenfolge: gefahrene Ankunftsfolge gegen die zugeordnete Soll-NUMMER je Tour.
# Anders als Kapitel 5 (gegen die PRC-TP-Nummer) vergleicht das gegen den tatsächlichen Soll-Plan,
# was bei re-disponierten Touren abweicht, weil PRC dort nach Ausführung umnummeriert.
treffer = vergleich[vergleich["ausfuehrung"] == "Treffer"].dropna(subset=["ist_ankunft", "soll_nummer"])

reihenfolge_plan = {}
for t_nr, g in treffer.groupby("tour_nr"):
    if len(g) < 2:
        continue
    folge = g.sort_values("ist_ankunft")["soll_nummer"].tolist()
    reihenfolge_plan[t_nr] = folge != sorted(folge)

plan_abw = pd.Series(reihenfolge_plan)
ist_redisp = pd.Series(plan_abw.index.isin(redisp_touren), index=plan_abw.index)
print(f"Vergleichbare Touren (>= 2 zugeordnete Stopps): {len(plan_abw)}")
print(f"Reihenfolge-Abweichung gegen den Soll-Plan: {int(plan_abw.sum())}")
print(f"  davon ohne Re-Disposition (eigenmächtig umsortiert): {int((plan_abw & ~ist_redisp).sum())}")
print(f"  davon mit Re-Disposition (geplante Umplanung):       {int((plan_abw & ist_redisp).sum())}")
print(f"\nZum Vergleich Kapitel 5 (gegen die PRC-TP-Nummer): {int(touren['abweichung'].sum())}")

Vergleichbare Touren (>= 2 zugeordnete Stopps): 189
Reihenfolge-Abweichung gegen den Soll-Plan: 89
  davon ohne Re-Disposition (eigenmächtig umsortiert): 52
  davon mit Re-Disposition (geplante Umplanung):       37

Zum Vergleich Kapitel 5 (gegen die PRC-TP-Nummer): 57


In Kapitel 5 haben wir die gefahrene Reihenfolge gegen die PRC-eigene Stopp-Nummer verglichen. Das beantwortet aber noch nicht die eigentliche Projektfrage, denn die geplante Reihenfolge steht nur in den Soll-Daten. Bei re-disponierten Touren nummeriert PRC nach der Ausführung, während der Soll den ursprünglichen Plan trägt. Gegen die PRC-Nummer bleibt eine geplante Umstellung deshalb unsichtbar. Jetzt, wo der geografische Verbund jedem gefahrenen Stopp seine Soll-NUMMER zugeordnet hat, lässt sich die gefahrene Ankunftsfolge direkt gegen die geplante Reihenfolge prüfen.

Gegen den Soll-Plan weichen 89 der 189 vergleichbaren Touren in ihrer Reihenfolge ab, deutlich mehr als die 57 gegen die PRC-Nummer. Die Differenz sind genau die Touren, deren gefahrene Folge zur umnummerierten PRC-Fassung passt, aber nicht zum ursprünglichen Plan. Die 89 zerfallen in zwei Arten, 52 Touren weichen ab, ohne je re-disponiert worden zu sein, hier hat der Fahrer die geplante Reihenfolge eigenmächtig umgestellt. Die anderen 37 Touren weichen ab und sind zugleich re-disponiert, hier verschob die Umplanung selbst die Reihenfolge. Beide sind echte Abweichungen vom Plan, aber sie haben verschiedene Ursachen. Die Aufteilung in eigenmächtige und re-dispositionsbedingte Abweichung greift den Gedanken der Schnittmengen-Prüfung aus Kapitel 5 wieder auf, beruht jetzt aber auf dem vollständigen geografischen Verbund statt auf der PRC-internen Basis-Version.

In [72]:
v = vergleich.copy()

# Standzeit nur aus verwertbaren Verweildauern (>= 5 min); Schub-Stopps bleiben draußen
v["verwertbare_verweil_min"] = v["ist_verweil_min"].where(v["ist_verweil_min"] >= 5)

# Re-Dispositions-Flag je Tour: trägt sie mindestens einen versionierten Stopp?
version = stat["StationID"].str.split("-").str[0].str.split("_").str[1].fillna("1").astype(int)
redisp = version.groupby(stat["tour_nr"]).max().gt(1).rename("redisponiert")

tour = v.groupby("tour_nr").agg(
    n_geplant              = ("soll_ankunft",            lambda s: s.notna().sum()),
    n_gefahren             = ("prc_stopp",               lambda s: s.notna().sum()),
    zeit_abw_median_min    = ("zeit_abw_min",            "median"),
    n_standzeit_verwertbar = ("verwertbare_verweil_min", lambda s: s.notna().sum()),
    standzeit_summe_min    = ("verwertbare_verweil_min", "sum"),
)
# Plan-bezogene Reihenfolge-Abweichung (Kapitel 7) und Re-Disposition anfügen
tour = tour.join(plan_abw.rename("reihenfolge_abw")).join(redisp)
tour["reihenfolge_abw"] = tour["reihenfolge_abw"].fillna(False)
tour["redisponiert"]    = tour["redisponiert"].fillna(False)
tour["zeit_abw_median_min"] = tour["zeit_abw_median_min"].round(1)
tour["standzeit_summe_min"] = tour["standzeit_summe_min"].round(0)

print(f"Touren gesamt: {len(tour)}")
print(f"  mit Reihenfolge-Abweichung (gegen Soll-Plan): {int(tour['reihenfolge_abw'].sum())}")
print(f"  re-disponiert: {int(tour['redisponiert'].sum())}")
print(f"  ohne jede PRC-Stationsmeldung: {int((tour['n_gefahren'] == 0).sum())}")
tour.sort_values("n_geplant", ascending=False).head(10).rename(columns={
    "n_geplant": "Stopps geplant (CIS)", "n_gefahren": "Stopps in PRC erfasst",
    "zeit_abw_median_min": "Zeitabweichung Median (min)",
    "n_standzeit_verwertbar": "Standzeit-Stopps verwertbar",
    "standzeit_summe_min": "Standzeit-Summe (min)",
    "reihenfolge_abw": "Reihenfolge-Abweichung", "redisponiert": "Re-disponiert"})

Touren gesamt: 203
  mit Reihenfolge-Abweichung (gegen Soll-Plan): 89
  re-disponiert: 71
  ohne jede PRC-Stationsmeldung: 4


,Stopps geplant (CIS),Stopps in PRC erfasst,Zeitabweichung Median (min),Standzeit-Stopps verwertbar,Standzeit-Summe (min),Reihenfolge-Abweichung,Re-disponiert
tour_nr,,,,,,,
H/42510,20,9,21.0,2,62.0,True,False
H/42450,16,15,-49.2,6,161.0,True,True
H/42480,16,14,-147.3,3,121.0,True,True
H/42488,16,14,-99.9,4,98.0,True,True
H/42513,16,15,-60.3,7,348.0,False,True
H/42459,15,7,199.4,5,234.0,True,False
H/42525,15,13,-90.1,7,138.0,True,False
H/42492,15,14,72.5,9,370.0,True,True
H/42462,15,12,-34.2,1,74.0,True,True


Jede Tour steht jetzt in einer Zeile. Die Spalten Stopps geplant und Stopps in PRC erfasst stellen die geplanten Stopps denen gegenüber, für die PRC eine Ankunft meldet. Eine Lücke zwischen beiden bedeutet nicht zwangsläufig einen ausgelassenen Stopp, sondern kann auch heißen, dass die Box dort keine Ankunft gesendet hat oder der Stopp im Sekundenschub liegt. Reihenfolge-Abweichung trägt die plan-bezogene Abweichung von oben und Re-disponiert die 71 re-disponierten Touren. Zeitabweichung Median fasst Verfrühung oder Verspätung der Tour zusammen, bewusst als Median und nicht als Mittelwert, damit einzelne Ausreißer die Tour-Kennzahl nicht verzerren. Standzeit-Summe summiert nur die verwertbaren Verweildauern, und Standzeit-Stopps verwertbar sagt dazu, aus wie vielen Stopps diese Summe stammt. Die beiden Spalten sind nur gemeinsam aussagekräftig, weil eine hohe Summe aus wenigen langen oder vielen kurzen Stopps stammen kann.

Die beiden letzten Spalten zusammengelesen ergeben die vier möglichen Fälle aus Reihenfolge und Re-Disposition. Eine Tour mit Reihenfolge-Abweichung, aber ohne Re-Disposition wurde vom Fahrer eigenmächtig umsortiert. Eine Tour mit beidem hat ihre Reihenfolge durch die Umplanung verschoben und eine re-disponierte Tour ohne Reihenfolge-Abweichung wurde zwar umgeplant, am Ende aber doch in der zum Plan passenden Folge gefahren. Genau diese Trennung ist die plan-bezogene Aufteilung der 89 in 52 eigenmächtige und 37 re-dispositionsbedingte Abweichungen.

# 8 – EMR-Geofencing: Standzeit aus der dichten GPS-Spur

Wir haben gezeigt, dass die PRC-Position im Stand schweigt und die Standzeit nur für rund die Hälfte der Stopps trägt. Die EMR-Spur pingt allerdings auch im Stand weiter (Median 26 Punkte im Stopp-Fenster) und ist damit die Quelle, um die Standzeit zu vervollständigen. EMR trägt keine Tour- oder Stopp-Kennung, nur Fahrzeug, Zeit und Position. Daher verknüpfen wir über Fahrzeug und Tag und ordnen anschließend geografisch zu.

Das Verfahren hat drei Schritte. Erstens erkennen wir aus der reinen GPS-Spur die Aufenthalte in Form von zusammenhängenden Strecken, in denen das Fahrzeug innerhalb eines engen Radius bleibt. Ein enger Radius trennt nahe beieinander liegende Stopps sauber. Gegen die Fragmentierung, also wenn ein Fahrzeug auf einem großen Gelände driftet, führen wir zwei Stücke mit kurzer Lücke am selben Ort wieder zusammen. Zweitens ordnen wir jeden Aufenthalt dem nächsten Soll-Stopp derselben Tour zu, mit derselben 500-m-Grenze wie der Kapitel-7-Verbund, weil die Soll-Geocodes bis zu einige hundert Meter neben dem tatsächlichen Halteplatz liegen. Als drittes prüfen wir die Geofence-Standzeit dort gegen, wo PRC eine verwertbare Standzeit hat.

An den Basis-Standorten, von denen die Touren starten, parken die Fahrzeuge vor und nach der Tour. Das zeigt eine nenenswerte Grenze, denn dort fängt der Geofence Parken statt Standzeit und beides lässt sich nicht trennen. Wir erkennen die Basen daran, dass sie als erster Stopp einer Tour auftreten und nehmen ihre Stopps aus der Standzeit-Auswertung aus.

In [73]:
# EMR-Daten laden
emr = pd.read_pickle("../data/processed/istdaten_emr_clean.pkl")
emr["tag"] = emr["Meldungszeit"].dt.date

# Basen = Orte, die je als erster Stopp (NUMMER 1) einer Tour auftreten -> dort wird geparkt
basen = set(soll.loc[soll["NUMMER"] == 1, "NAME1"].unique())
print("Basis-Standorte (Tour-Startorte):", basen)

def detect_aufenthalte(sp, R=150, merge_gap_min=5, min_min=3):
    """Aufenthalte aus einer GPS-Spur: zusammenhängende Strecken innerhalb von R Metern.
    Kurz getrennte Stücke am selben Ort werden zusammengeführt (gegen Fragmentierung)."""
    lat = sp["Breitengrad"].values; lon = sp["Längengrad"].values; t = sp["Meldungszeit"].values; n = len(sp)
    # 1) Rohcluster: Punkt bleibt im Cluster, solange er < R vom Cluster-Anker liegt
    roh = []; i = 0
    while i < n:
        j = i
        while j + 1 < n and haversine(lat[i], lon[i], lat[j+1], lon[j+1]) <= R:
            j += 1
        roh.append([i, j]); i = j + 1
    # 2) Merge: aufeinanderfolgende Cluster mit kurzer Lücke am selben Ort zusammenführen
    merged = [list(roh[0])]
    for a, b in roh[1:]:
        pa, pb = merged[-1]
        luecke = (t[a] - t[pb]) / np.timedelta64(1, "m")
        c_prev = (lat[pa:pb+1].mean(), lon[pa:pb+1].mean()); c_cur = (lat[a:b+1].mean(), lon[a:b+1].mean())
        if luecke <= merge_gap_min and haversine(*c_prev, *c_cur) <= R:
            merged[-1][1] = b
        else:
            merged.append([a, b])
    # 3) Aufenthalte ab Mindestdauer
    auf = []
    for a, b in merged:
        dauer = (t[b] - t[a]) / np.timedelta64(1, "m")
        if dauer >= min_min:
            auf.append({"start": pd.Timestamp(t[a]), "ende": pd.Timestamp(t[b]),
                        "dauer_min": round(dauer, 1), "lat": lat[a:b+1].mean(), "lon": lon[a:b+1].mean()})
    return pd.DataFrame(auf)

# Demonstration an H/42378 (Fahrzeug OD-TS-8044)
g = soll[soll["TOURNR"] == "H/42378"].sort_values("ANKUNFT")
sp = emr[(emr["LKW_KENNZ"] == g["LKW_KENNZ"].iloc[0]) & (emr["tag"] == g["ANKUNFT"].iloc[0].date())].sort_values("Meldungszeit").reset_index(drop=True)
auf = detect_aufenthalte(sp)
print(f"\nSpur OD-TS-8044 am {g['ANKUNFT'].iloc[0].date()}: {len(sp)} Punkte, {len(auf)} Aufenthalte")
auf[["start", "ende", "dauer_min"]]

Basis-Standorte (Tour-Startorte): {'Greif Packaging Germany GmbH', 'HTH Hanse GmbH'}

Spur OD-TS-8044 am 2026-03-02: 538 Punkte, 12 Aufenthalte


,start,ende,dauer_min
0,2026-03-02 01:00:30,2026-03-02 06:20:34,320.1
1,2026-03-02 07:37:06,2026-03-02 07:42:31,5.4
2,2026-03-02 07:43:27,2026-03-02 07:48:31,5.1
3,2026-03-02 08:20:33,2026-03-02 08:55:13,34.7
4,2026-03-02 08:56:01,2026-03-02 09:03:53,7.9
5,2026-03-02 09:13:36,2026-03-02 09:16:36,3.0
6,2026-03-02 09:40:29,2026-03-02 09:44:15,3.8
7,2026-03-02 10:16:40,2026-03-02 10:20:41,4.0
8,2026-03-02 10:25:00,2026-03-02 10:32:00,7.0
9,2026-03-02 11:22:57,2026-03-02 11:34:39,11.7


Die Spur von OD-TS-8044 zerfällt in zwölf Aufenthalte. Der 320-min-Block am Morgen (01:00–06:20) ist das Parken am Depot vor Tourbeginn, der 284-min-Block am Ende (11:58–16:42) das Parken nach der Rückkehr und beide an der Basis HTH Hanse. Dazwischen liegen die Kundenstopps mit wenigen bis gut zehn Minuten. Drei Aufenthalte zwischen 08:20 und 09:16 lassen sich keinem Soll-Stopp zuordnen, darunter ein längerer von rund 35 Minuten. Der Ursprung, also ob Pause, Stau oder ein ungeplanter Halt, geht aus der Spur allein nicht hervor und bleibt offen. Die Aufenthalte sind also da und müssen den geplanten Stopps zugeordnet werden unter Berücksichtigung der Basen.

In [74]:
# Aufenthalte je Tour erkennen und den Soll-Stopps geografisch zuordnen (500 m wie Kapitel 7)
spuren = {k: v.sort_values("Meldungszeit").reset_index(drop=True) for k, v in emr.groupby(["LKW_KENNZ", "tag"])}

zeilen = []
for tnr, g in soll.groupby("TOURNR"):
    g = g.sort_values("ANKUNFT"); kennz = g["LKW_KENNZ"].iloc[0]
    if pd.isna(kennz):
        continue
    sp = spuren.get((kennz, g["ANKUNFT"].iloc[0].date()))
    if sp is None or len(sp) < 5:
        continue
    auf = detect_aufenthalte(sp)
    frei = list(g.index)
    for _, r in auf.sort_values("start").iterrows():
        if not frei:
            break
        d = haversine(r["lat"], r["lon"], g.loc[frei, "GEOY"].values, g.loc[frei, "GEOX"].values)
        k = int(d.argmin())
        if d[k] <= 500:
            idx = frei.pop(k)
            zeilen.append((tnr, int(g.loc[idx, "NUMMER"]), g.loc[idx, "NAME1"],
                           round(r["dauer_min"], 1), round(d[k]), g.loc[idx, "NAME1"] in basen))
emr_standzeit = pd.DataFrame(zeilen, columns=["tour_nr", "soll_nummer", "NAME1", "emr_verweil_min", "dist_m", "ist_basis"])

print(f"Soll-Stopps mit EMR-Aufenthalt: {len(emr_standzeit)}")
print(f"  an Kunden (verwertbar):        {int((~emr_standzeit['ist_basis']).sum())}")
print(f"  an Basen (Parken, ausgenommen): {int(emr_standzeit['ist_basis'].sum())}")
emr_standzeit[emr_standzeit["tour_nr"] == "H/42378"].sort_values("soll_nummer")

Soll-Stopps mit EMR-Aufenthalt: 1279
  an Kunden (verwertbar):        614
  an Basen (Parken, ausgenommen): 665


,tour_nr,soll_nummer,NAME1,emr_verweil_min,dist_m,ist_basis
20,H/42378,1,HTH Hanse GmbH,320.1,59,True
22,H/42378,2,Kälte Klima Nord GbR,4.0,25,False
23,H/42378,3,Klima-Schröder GmbH,7.0,331,False
21,H/42378,4,Hustede Haustechnik,5.4,366,False
24,H/42378,5,K+J Haustechnik GmbH,11.7,39,False
25,H/42378,6,HTH Hanse GmbH,283.7,36,True


In [75]:
# Validierung: EMR-Kundenstandzeit gegen die verwertbare PRC-Standzeit aus Kapitel 7
val = emr_standzeit.merge(vergleich[["tour_nr", "soll_nummer", "ist_verweil_min"]],
                          on=["tour_nr", "soll_nummer"], how="left")
kunden = val[~val["ist_basis"] & (val["emr_verweil_min"] >= 5)]
beide = kunden[kunden["ist_verweil_min"] >= 5].copy()
beide["diff"] = beide["emr_verweil_min"] - beide["ist_verweil_min"]

print(f"EMR-Standzeit an Kunden (verwertbar, >=5 min): {len(kunden)}")
print(f"  davon auch mit verwertbarer PRC-Standzeit (Validierung): {len(beide)}")
print(f"  davon ohne verwertbare PRC-Standzeit (EMR füllt die Lücke): {len(kunden) - len(beide)}")
print(f"\nEMR−PRC Differenz über die {len(beide)} gemeinsamen Stopps:")
print(f"  Median {beide['diff'].median():.1f} min | 10/90-Perzentil {beide['diff'].quantile(.1):.0f}/{beide['diff'].quantile(.9):.0f} min")
print(f"  innerhalb ±5 min: {(beide['diff'].abs() <= 5).mean()*100:.0f}%  |  innerhalb ±10 min: {(beide['diff'].abs() <= 10).mean()*100:.0f}%")

EMR-Standzeit an Kunden (verwertbar, >=5 min): 538
  davon auch mit verwertbarer PRC-Standzeit (Validierung): 303
  davon ohne verwertbare PRC-Standzeit (EMR füllt die Lücke): 235

EMR−PRC Differenz über die 303 gemeinsamen Stopps:
  Median 2.4 min | 10/90-Perzentil -14/15 min
  innerhalb ±5 min: 54%  |  innerhalb ±10 min: 71%


An 303 Kundenstopps liegt sowohl eine EMR- als auch eine verwertbare PRC-Standzeit vor, was den direkten Abgleich zweier unabhängiger Quellen erlaubt, der server-seitig aufgezeichneten PRC-Meldung und der Fahrer-App-Spur aus EMR. Sie stimmen mit einem Median von 2,4 Minuten Differenz,einem 10- und 90-Perzentil von −14 bis +15 Minuten, 54 Prozent innerhalb von fünf und 71 Prozent innerhalb von zehn Minuten weitgehend überein. Die EMR-Standzeit liegt im Median gut zwei Minuten über der PRC-Standzeit. Eine mögliche Erklärung ist, dass die EMR-Spur bis zur physischen Abfahrt weiterpingt, während der PRC-Abfahrt-Status zu einem etwas anderen Zeitpunkt gesetzt wird. Der Versatz ist mit gut zwei Minuten klein, die verbleibende Streuung zeigt sich in den Perzentilen. Diese Übereinstimmung ist die Validierung, dass das Geofence-Verfahren im Rahmen dieser Streuung dasselbe misst wie der Stationsstatus.

Der eigentliche Gewinn liegt in den übrigen Stopps. Insgesamt bekommen 538 Kundenstopps eine verwertbare EMR-Standzeit ab fünf Minuten, davon 235, für die PRC keine hatte. Das ist genau ein Teil der Schub-Hälfte aus Kapitel 4, die als Sekundenschub unbrauchbar war und der nun über die EMR-Spur ergänzt werden kann.

In [76]:
em = emr_standzeit.loc[~emr_standzeit["ist_basis"], ["tour_nr", "soll_nummer", "emr_verweil_min"]]
vergleich = vergleich.merge(em, on=["tour_nr", "soll_nummer"], how="left")
vergleich["ist_basis"] = vergleich["NAME1"].isin(basen)

# Finale Standzeit: an Kunden EMR bevorzugt, PRC als Rückfall; Basis-Stopps ausgenommen
vergleich["standzeit_min"] = vergleich["emr_verweil_min"].where(vergleich["emr_verweil_min"].notna(), vergleich["ist_verweil_min"])
vergleich["standzeit_quelle"] = np.select(
    [vergleich["ist_basis"], vergleich["emr_verweil_min"].notna(), vergleich["ist_verweil_min"].notna()],
    ["Basis", "EMR", "PRC"], default="fehlt")
vergleich.loc[vergleich["ist_basis"], "standzeit_min"] = np.nan

# Plausibilitätsgrenze (neu)
unplausibel = vergleich["standzeit_min"] > 360
vergleich.loc[unplausibel, "standzeit_quelle"] = "unplausibel"
vergleich.loc[unplausibel, "standzeit_min"] = np.nan

# Standzeit-Abweichung neu, nur für verwertbare Kundenstandzeit (>= 5 min)
verw = (vergleich["standzeit_min"] >= 5) & ~vergleich["ist_basis"]
vergleich["standzeit_abw_min"] = (vergleich["standzeit_min"] - vergleich["soll_standzeit_min"]).round(1)
vergleich.loc[~verw, "standzeit_abw_min"] = np.nan

print("Standzeit-Quelle je Stopp:")
print(vergleich["standzeit_quelle"].value_counts().to_string())
print(f"\nKundenstopps mit verwertbarer Standzeit (>= 5 min): {int(verw.sum())}")
print("Standzeit-Abweichung Ist−Soll (min, verwertbar):")
print(vergleich.loc[verw, "standzeit_abw_min"].describe(percentiles=[.1, .5, .9]).round(1).to_string())
vergleich[vergleich["tour_nr"] == "H/42378"].sort_values("soll_nummer")[
    ["soll_nummer", "NAME1", "soll_standzeit_min", "ist_verweil_min",
     "emr_verweil_min", "standzeit_min", "standzeit_quelle", "standzeit_abw_min"]].rename(columns={
    "soll_nummer": "Soll-Stopp", "NAME1": "Kunde",
    "soll_standzeit_min": "Soll-Standzeit (min)", "ist_verweil_min": "PRC-Verweildauer (min)",
    "emr_verweil_min": "EMR-Verweildauer (min)", "standzeit_min": "finale Standzeit (min)",
    "standzeit_quelle": "Quelle", "standzeit_abw_min": "Standzeit-Abweichung (min)"})

Standzeit-Quelle je Stopp:
standzeit_quelle
Basis          693
EMR            614
PRC            283
fehlt          240
unplausibel      2

Kundenstopps mit verwertbarer Standzeit (>= 5 min): 631
Standzeit-Abweichung Ist−Soll (min, verwertbar):
count    540.0
mean      10.8
std       40.7
min     -334.4
10%      -16.6
50%       -3.5
90%       58.9
max      335.3


,Soll-Stopp,Kunde,Soll-Standzeit (min),PRC-Verweildauer (min),EMR-Verweildauer (min),finale Standzeit (min),Quelle,Standzeit-Abweichung (min)
27,1.0,HTH Hanse GmbH,0.0,87.3,NaN,NaN,Basis,NaN
28,2.0,Kälte Klima Nord GbR,15.0,2.9,4.0,4.0,EMR,NaN
29,3.0,Klima-Schröder GmbH,15.0,5.0,7.0,7.0,EMR,-8.0
30,4.0,Hustede Haustechnik,15.0,1.2,5.4,5.4,EMR,-9.6
31,5.0,K+J Haustechnik GmbH,15.0,7.4,11.7,11.7,EMR,-3.3
32,6.0,HTH Hanse GmbH,0.0,5.5,NaN,NaN,Basis,NaN


In [77]:
# Tour-Standzeit mit der finalen (EMR-basierten) Kundenstandzeit auffrischen
sz_min = pd.to_numeric(vergleich["standzeit_min"], errors="coerce")
verwertbar = sz_min.where((sz_min >= 5) & ~vergleich["ist_basis"])
agg = verwertbar.groupby(vergleich["tour_nr"]).agg(["sum", "count"])

tour["standzeit_summe_min"] = agg["sum"].round(0).reindex(tour.index).fillna(0)
tour["n_standzeit_verwertbar"] = agg["count"].reindex(tour.index).fillna(0).astype(int)

print("Tour-Standzeit aufgefrischt mit der finalen Kundenstandzeit.")
print(f"Touren mit mindestens einer verwertbaren Kundenstandzeit: {int((tour['n_standzeit_verwertbar'] > 0).sum())}")
tour.sort_values("standzeit_summe_min", ascending=False).head(8).rename(columns={
    "n_geplant": "Stopps geplant", "n_gefahren": "Stopps in PRC erfasst",
    "zeit_abw_median_min": "Zeitabweichung Median (min)",
    "n_standzeit_verwertbar": "Standzeit-Stopps verwertbar",
    "standzeit_summe_min": "Standzeit-Summe (min)",
    "reihenfolge_abw": "Reihenfolge-Abweichung", "redisponiert": "Re-disponiert"})

Tour-Standzeit aufgefrischt mit der finalen Kundenstandzeit.
Touren mit mindestens einer verwertbaren Kundenstandzeit: 192


,Stopps geplant,Stopps in PRC erfasst,Zeitabweichung Median (min),Standzeit-Stopps verwertbar,Standzeit-Summe (min),Reihenfolge-Abweichung,Re-disponiert
tour_nr,,,,,,,
H/42532,7,7,79.2,4,563.0,True,True
H/42504,11,8,-1.6,5,444.0,False,True
H/42468,7,6,-13.2,4,443.0,True,True
H/42630,13,13,42.8,6,441.0,False,True
H/42631,10,9,127.8,4,438.0,True,True
H/42557,8,9,-14.1,4,435.0,True,True
H/42492,15,14,72.5,7,388.0,True,True
H/42517,14,11,-275.5,6,352.0,True,False


Damit ist die Standzeit-Spalte vollständig. An den Kundenstopps steht die EMR-Standzeit (614 Stopps), als Rückfall die PRC-Standzeit, wo EMR keinen Aufenthalt fand (283 Stopps), und die 693 Basis-Stopps sind ausgenommen. Für 240 Kundenstopps liefert keine der beiden Quellen eine Standzeit, sei es wegen zu kurzen Halts oder fehlender Spur.

Zwei Werte sind als unplausibel markiert und nicht verwendet. Sie stammen aus dem PRC-Rückfall und gehen auf die verschleppten Zwei-Tage-Abfahrten aus Kapitel 4 zurück. Die Grenze von sechs Stunden trifft ausschließlich solche Rückfallwerte, denn die validierten EMR-Standzeiten bleiben alle darunter, mit einem Höchstwert von 335 Minuten an einem einzelnen Kundenstopp. Die Schwelle ist damit ein Sicherheitsnetz gegen die bekannten Probleme und berührt keinen echten EMR-Wert.

Insgesamt tragen 631 Kundenstopps eine verwertbare Standzeit ab fünf Minuten, also die 614 aus EMR zusammen mit den PRC-Rückfallwerten, die ebenfalls die Fünf-Minuten-Schwelle erreichen. Die Standzeiten kommen also überwiegend aus der dichten EMR-Spur und damit auch dort, wo PRC nur einen Sekundenschub hatte. Über die 540 Stopps mit geplanter und verwertbarer Ist-Standzeit liegt die Abweichung im Median bei −3,5 Minuten, die Fahrzeuge stehen im Mittel also kürzer als geplant. Der rechte Rand ist aber lang, mit einem 90-Perzentil von rund 59 Minuten. Das sind die tatsächlichen Standzeit-Überschreitungen bei einzelnen Kunden, die die Analyse sichtbar macht. Die Standzeit-Quelle bleibt je Stopp dokumentiert, sodass im Dashboard nachvollziehbar ist, woher jeder Wert stammt.

Damit liegen alle vier Ist-Kennzahlen vor. Stopp-Reihenfolge und Re-Disposition aus PRC, Zeitabweichung aus PRC, Standzeit aus EMR mit PRC als Rückfall und gegenseitiger Validierung, und die gefahrene Route aus der dichten Spur.

## 8.1 Robustheit des Geofencing-Radius

Der Radius von 150 Metern in detect_aufenthalte ist ein gesetzter Parameter. Wir prüfen, wie empfindlich das Ergebnis auf ihn reagiert, indem wir das Geofencing für mehrere Radien wiederholen und je Radius drei Größen ansehen: wie viele Kundenstopps einen Aufenthalt bekommen, wie hoch die mediane Kundenstandzeit ist und wie gut die EMR-Standzeit mit der unabhängigen PRC-Standzeit übereinstimmt.

In [78]:
# Robustheit gegenüber dem Radius: Geofencing für mehrere Radien wiederholen.
# prc_verweil je Stopp aus der verwertbaren PRC-Standzeit (für den Übereinstimmungs-Check)
prc_verweil = (vergleich[vergleich["ist_verweil_min"] >= 5]
               .set_index(["tour_nr", "soll_nummer"])["ist_verweil_min"])

print(f"{'Radius':>7} | {'Kundenstopps':>12} | {'Median Standzeit':>16} | {'mittl.abs.Abw zu PRC':>20}")
for R in [100, 125, 150, 175, 200]:
    zeilen = []
    for tnr, g in soll.groupby("TOURNR"):
        g = g.sort_values("ANKUNFT"); kennz = g["LKW_KENNZ"].iloc[0]
        if pd.isna(kennz):
            continue
        sp = spuren.get((kennz, g["ANKUNFT"].iloc[0].date()))
        if sp is None or len(sp) < 5:
            continue
        auf = detect_aufenthalte(sp, R=R); frei = list(g.index)
        for _, r in auf.sort_values("start").iterrows():
            if not frei:
                break
            d = haversine(r["lat"], r["lon"], g.loc[frei, "GEOY"].values, g.loc[frei, "GEOX"].values)
            k = int(d.argmin())
            if d[k] <= 500:
                idx = frei.pop(k)
                if g.loc[idx, "NAME1"] not in basen:
                    zeilen.append((tnr, int(g.loc[idx, "NUMMER"]), r["dauer_min"]))
    e = pd.DataFrame(zeilen, columns=["tour_nr", "soll_nummer", "emr"]).query("emr >= 5")
    m = e.join(prc_verweil, on=["tour_nr", "soll_nummer"]).dropna()
    abw = (m["emr"] - m["ist_verweil_min"]).abs().median()
    print(f"{R:>6}m | {len(e):>12} | {e['emr'].median():>14.1f} m | {abw:>18.1f} m")

 Radius | Kundenstopps | Median Standzeit | mittl.abs.Abw zu PRC
   100m |          497 |           16.9 m |                7.2 m
   125m |          520 |           19.4 m |                5.0 m
   150m |          538 |           21.9 m |                4.3 m
   175m |          551 |           22.4 m |                4.3 m
   200m |          558 |           23.1 m |                4.4 m


Das Ergebnis trägt die Wahl von 150 Metern. Die Zahl der erkannten Kundenstopps wächst über den Bereich von 100 bis 200 Metern nur leicht, von 497 auf 558. Welche Aufenthalte erkannt und welchen Soll-Stopps sie zugeordnet werden, hängt also nur schwach vom Radius ab. Die mediane Standzeit steigt dagegen spürbar mit dem Radius, von rund 17 Minuten bei 100 Metern auf rund 23 bei 200, weil ein weiterer Radius einen Teil der An- und Abfahrt mit in den Aufenthalt zieht. Dieses Problem fangen wir über die Validierung gegen PRC ab. Die mittlere absolute Abweichung zwischen EMR- und PRC-Standzeit ist bei engem Radius am höchsten (7,2 Minuten bei 100 Metern), fällt zu 150 Metern hin auf rund 4,3 Minuten und bleibt darüber praktisch konstant (4,3 bei 175, 4,4 bei 200). Größere Radien verbessern die Übereinstimmung also nicht mehr, vergrößern aber die Gefahr, benachbarte Stopps zu verschmelzen. Die 150 Meter liegen damit genau an der Stelle, ab der die EMR-Standzeit am engsten mit der unabhängigen PRC-Messung übereinstimmt. Eng genug, um Stopps zu trennen, aber noch weit genug, um den Aufenthalt vollständig zu erfassen.

# Export für das Dashboard

Zum Abschluss schreiben wir drei Tabellen, die das Dashboard direkt laden kann. Die Stopp-Tabelle trägt je Zeile einen Stopp mit seinen Koordinaten und allen Ist-Kennzahlen. Sie umfasst 1832 Zeilen, nämlich die 1577 geplanten Soll-Stopps plus die 255 gefahrenen Stopps, die keinem Plan-Stopp zugeordnet werden konnten und als ungeplante Halte erhalten bleiben. Die Tour-Tabelle verdichtet das auf eine Zeile je Tour und trägt zusätzlich Fahrzeug und Tag, damit das Dashboard die passende Spur findet. Die Spur-Tabelle enthält die GPS-Punkte der dichten EMR-Spur für die Kartendarstellung der gefahrenen Route, da sie mit Abstand die durchgehendste Quelle für den Wegverlauf darstellt. Alle drei werden als Pickle gespeichert, damit die in der Analyse gesetzten Datentypen erhalten bleiben.

In [79]:
OUT = Path.cwd().parent / "data" / "processed"

# 1) Stopp-Tabelle: Soll mit Koordinaten + alle Ist-Kennzahlen je Stopp
stop_keys = soll[["TOURNR", "NUMMER", "GEOY", "GEOX"]].rename(columns={"TOURNR": "tour_nr", "NUMMER": "soll_nummer"})
stops_dash = vergleich.merge(stop_keys, on=["tour_nr", "soll_nummer"], how="left")[[
    "tour_nr", "soll_nummer", "NAME1", "GEOY", "GEOX", "ausfuehrung",
    "soll_ankunft", "ist_ankunft", "zeit_abw_min",
    "soll_standzeit_min", "standzeit_min", "standzeit_abw_min", "standzeit_quelle", "ist_basis"]]

# 2) Tour-Tabelle: Kennzahlen + Fahrzeug/Tag (damit die App die EMR-Spur findet)
tmap = (soll.sort_values("ANKUNFT").groupby("TOURNR")
        .agg(lkw_kennz=("LKW_KENNZ", "first"), tag=("ANKUNFT", lambda s: s.iloc[0].date())))
tour_dash = tour.join(tmap)

# 3) EMR-Spur, getrimmt auf das fürs Zeichnen Nötige
spur_dash = emr[["LKW_KENNZ", "Meldungszeit", "tag", "Breitengrad", "Längengrad"]].copy()

stops_dash.to_pickle(OUT / "dash_stops.pkl")
tour_dash.to_pickle(OUT / "dash_touren.pkl")
spur_dash.to_pickle(OUT / "dash_spur.pkl")

print("Exportiert nach data/processed/:")
print(f"  dash_stops.pkl  {stops_dash.shape}")
print(f"  dash_touren.pkl {tour_dash.shape}")
print(f"  dash_spur.pkl   {spur_dash.shape}")
print("\nStopp-Tabelle, Beispiel H/42378:")
stops_dash[stops_dash["tour_nr"] == "H/42378"].sort_values("soll_nummer")

Exportiert nach data/processed/:
  dash_stops.pkl  (1832, 14)
  dash_touren.pkl (203, 9)
  dash_spur.pkl   (149119, 5)

Stopp-Tabelle, Beispiel H/42378:


,tour_nr,soll_nummer,NAME1,GEOY,GEOX,ausfuehrung,soll_ankunft,ist_ankunft,zeit_abw_min,soll_standzeit_min,standzeit_min,standzeit_abw_min,standzeit_quelle,ist_basis
27,H/42378,1.0,HTH Hanse GmbH,53.66737,9.99189,Treffer,2026-03-02 05:00:00,2026-03-02 04:51:33,-8.450000,0.0,NaN,NaN,Basis,True
28,H/42378,2.0,Kälte Klima Nord GbR,53.92675,9.51887,Treffer,2026-03-02 07:33:00,2026-03-02 10:16:53,163.883333,15.0,4.0,NaN,EMR,False
29,H/42378,3.0,Klima-Schröder GmbH,53.93040,9.49298,Treffer,2026-03-02 07:56:00,2026-03-02 10:25:59,149.983333,15.0,7.0,-8.0,EMR,False
30,H/42378,4.0,Hustede Haustechnik,53.59601,9.44293,Treffer,2026-03-02 09:41:00,2026-03-02 07:45:15,-115.750000,15.0,5.4,-9.6,EMR,False
31,H/42378,5.0,K+J Haustechnik GmbH,53.60195,10.02743,Treffer,2026-03-02 11:11:00,2026-03-02 11:24:38,13.633333,15.0,11.7,-3.3,EMR,False
32,H/42378,6.0,HTH Hanse GmbH,53.66737,9.99189,Treffer,2026-03-02 11:48:00,2026-03-02 11:59:49,11.816667,0.0,NaN,NaN,Basis,True


# 9 – Fazit: Tragfähigkeit der Datenbasis für das Dashboard

Aus drei getrennt konsolidierten Quellen ist eine integrierte Ist-Datenbasis entstanden, die vier Kennzahlen je Stopp und je Tour trägt. Die Re-Disposition stammt aus dem PRC-Stationsstatus, die Stopp-Reihenfolge und die Zeitabweichung aus dem geografischen Verbund von PRC-Ankünften und Soll-Plan, die Standzeit aus dem EMR-Geofencing mit PRC als Rückfall und Validierung, und die gefahrene Route aus der dichten EMR-Spur. Verknüpft wurden die Quellen über das normalisierte Kennzeichen, die Tour und die Geografie, da kein gemeinsamer technischer Schlüssel zwischen ihnen besteht.

## Warum die Basis valide und verlässlich ist

Der stärkste Beleg für die Verlässlichkeit ist, dass zwei unabhängig erhobene Quellen dort übereinstimmen, wo sie sich überlappen. Die Standzeit aus dem EMR-Geofencing und die aus dem PRC-Stationsstatus wurden an den 303 Stopps verglichen, für die beide einen verwertbaren Wert haben und sie stimmen mit einem Median von 2,4 Minuten Differenz und 71 Prozent innerhalb von zehn Minuten überein. Das ist keine Selbstbestätigung, denn die eine Quelle wird server-seitig von der Fahrzeugbox aufgezeichnet, die andere aus der Fahrer-App und beide entstehen unabhängig voneinander. Dass sie dasselbe messen, ist der tragende Validitätsnachweis der Standzeit.

Die zweite Säule ist, dass jede Schwellen- und Parameterentscheidung empirisch begründet und auf ihre Robustheit geprüft wurde. Die 500-Meter-Zuordnungsgrenze ist robust, weil sich die Zuordnung zwischen 400 und 600 Metern nur um zwölf Stopps ändert, die meisten Zuordnungen liegen mit einem Median von 87 Metern weit darunter. Der Geofencing-Radius von 150 Metern wurde über eine Sensitivitätsanalyse gerechtfertigt, die zeigt, dass die Stopp-Erkennung über den ganzen Bereich von 100 bis 200 Metern stabil bleibt und die mittlere Abweichung zur PRC-Standzeit ab 150 Metern bei rund vier Minuten einrastet, ohne dass größere Radien sie weiter senken. Die Plausibilitätsgrenzen sind fachlich gesetzt und in ihrer Wirkung transparent. Die Zwölf-Stunden-Grenze der Zeitabweichung trifft acht Extremfälle zwischen 14 und 70 Stunden, die Sechs-Stunden-Grenze der Standzeit nur zwei PRC-Rückfallwerte, während kein validierter EMR-Wert sie erreicht.

Die dritte Säule ist die Wahl des geografischen Verbunds gegenüber der Zuordnung über die Stopp-Nummer. Die Gegenprobe zeigt, dass die Nummern-Zuordnung bei 35 Prozent der Stopps den falschen Ort trifft, weil PRC bei Re-Disposition nach der Ausführung nummeriert und der Soll nach dem ursprünglichen Plan. Der geografische Verbund umgeht das und ordnet im Median 87 Meter genau zu. Die vierte Säule ist die durchgängige Bilanztreue. 1127 Treffer plus 450 nur geplante Stopps ergeben die 1577 Soll-Stopps, 1127 zugeordnete plus 255 nicht zuordenbare ergeben die 1382 gefahrenen Stopps, und auf Tour-Ebene bleiben es lückenlos 203 Touren. Es geht kein Stopp verloren und es entsteht keiner aus dem Nichts.

## Was die Schwachpunkte dieser Basis für das Dashboard sind

Der erste Vorbehalt betrifft die Vollständigkeit der Standzeit. Für 240 Kundenstopps liefert keine Quelle eine verwertbare Standzeit, weil der Halt zu kurz war oder die Spur fehlte. Das Dashboard muss diese Lücken als solche zeigen und darf sie nicht als Null-Standzeit deuten, sonst entstünde der falsche Eindruck eines sofortigen Weiterfahrens. Die mitexportierte Standzeit-Quelle je Stopp erlaubt genau diese Unterscheidung und sollte sichtbar bleiben.

Der zweite Vorbehalt betrifft die absolute Höhe der Standzeit. Die Sensitivitätsanalyse hat gezeigt, dass der Wert vom Geofencing-Radius abhängt und bei größerem Radius systematisch steigt. Die Standzeit ist damit kein punktgenauer Messwert, sondern eine modellabhängige Schätzung, deren Größenordnung durch die PRC-Validierung abgesichert ist, deren Minutenwert im Einzelfall aber eine Unsicherheit von rund vier Minuten trägt. Für aggregierte Aussagen und Vergleiche zwischen Touren ist das unkritisch, für die minutengenaue Bewertung eines einzelnen Stopps sollte man es wissen.

Der dritte Vorbehalt betrifft die nicht zuordenbaren Stopps und die vier Touren ohne PRC-Stationsstatus. 255 gefahrene Stopps ließen sich keinem Soll-Stopp zuordnen. Die vier Touren ohne PRC-Stationsmeldung sind nicht ungefahren: Ihre EMR-Spur belegt, dass sie stattfanden und die geplanten Stopps anfuhren, nur hat die PRC-Box keinen Stationsstatus gesendet. Das Dashboard sollte diese vier deshalb nicht als ungefahren, sondern als in PRC nicht erfasst kennzeichnen, da eine als ungefahren markierte, tatsächlich aber gefahrene Tour eine irreführende Aussage wäre.

Der vierte Vorbehalt betrifft die Reihenfolge-Abweichung. Dass die Soll-NUMMER die geplante Reihenfolge kodiert, ist belegt, denn bei allen 203 Touren stimmt die NUMMER-Sortierung mit der Sortierung nach geplanter Ankunftszeit überein. Die gefahrene Reihenfolge wird über den geografischen Verbund direkt gegen diese Soll-NUMMER geprüft, sodass sie unabhängig von der PRC-internen Nummerierung ist und auch geplante Umdispositionen als Abweichung vom ursprünglichen Plan erfasst. Die verbleibende Unsicherheit liegt in der geografischen Zuordnung selbst, die dort mehrdeutig wird, wo mehrere geplante Stopps am selben Ort liegen; diese Mehrdeutigkeit ist aber belegt folgenlos, weil sie zu 97 Prozent geografisch deckungsgleiche Stopps betrifft, deren Vertauschung die Reihenfolge nicht ändert.

## Gesamteinschätzung

Die Basis ist für ein Dashboard tragfähig, weil ihre Kernaussagen auf zwei unabhängigen Quellen ruhen, die sich gegenseitig bestätigen, und weil jede Modellentscheidung begründet und in ihrer Wirkung transparent ist. Ihre Grenzen sind nicht versteckt, sondern in den Daten selbst dokumentiert, vor allem über die Standzeit-Quelle je Stopp. Entscheidend für ein verlässliches Dashboard ist, dass es diese Grenzen mitführt statt sie zu glätten: fehlende Standzeiten als fehlend, ungeplante Halte als solche und die Herkunft jedes Standzeit-Werts nachvollziehbar. Bildet das Dashboard diese Ehrlichkeit der Basis ab, ist es eine valide Entscheidungsgrundlage.